# Football analytics: loader walkthrough

## Checkpoint 0: source and contract

Use `data/xDiyo_data`, seasons **2021/22 through 2024/25**. The league subset is still open. The [progress tracker](../IMPLEMENTATION_PROGRESS.md) holds the current agreement and proposed first function.

The opening cells display the **saved inspection snapshot from 15 September 2026**. Checkpoints 1–4 below demonstrate resolution, verified reading, match validation and simple loading with saved selections; no experiment population has been selected. After each function is implemented, we will add an import and a concrete call here, then discuss its actual code before continuing. Implementation stays in `src/xdiyo_analytics/`.

### This increment

1. Review the available publication counts and missingness.
2. Agree the resolver contract: an explicit season name becomes one pinned manifest reference.
3. Follow each implemented function below, ending with the simple loading API and its feedback checkpoint.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

EXPECTED_PYTHON = Path("C:/Users/luisi/Documents/Programming/Python/.misc314/Scripts/python.exe")
assert Path(sys.executable).resolve() == EXPECTED_PYTHON.resolve(), sys.executable
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "IMPLEMENTATION_PROGRESS.md").exists() else Path.cwd().parent
assert (PROJECT_ROOT / "IMPLEMENTATION_PROGRESS.md").is_file()
print("Interpreter:", sys.executable)
print("Project:", PROJECT_ROOT)

Interpreter: c:\Users\luisi\Documents\Programming\Python\.misc314\Scripts\python.exe
Project: c:\Users\luisi\Documents\Programming\Python\xDiyo


## What is available

The JSON below is evidence produced during the bounded inspection. Rendering that evidence is notebook presentation code; it does not implement export discovery or loading. One row in `inventory_view` is one league-season publication. Match counts refer to the saved fixture cohort, which may omit stages or fixtures outside the collection scope.

In [2]:
inspection = json.loads((PROJECT_ROOT / "docs/analytics/restart_inspection.json").read_text(encoding="utf-8"))
inventory_view = pd.DataFrame([
    {
        "league": item["scope"]["league_name"],
        "season": f'{item["scope"]["season_start"]}/{item["scope"]["season_end"]}',
        "competition_id": item["scope"]["league_id"],
        "season_id": item["scope"]["season_id"],
        "matches": item["matches_check"]["rows"],
        "null_kickoffs": item["matches_check"]["null_kickoffs"],
        "version": item["version"],
    }
    for item in inspection["inventory"] if "matches_check" in item
])
display(inventory_view.pivot(index="league", columns="season", values="matches"))
print(f"{len(inventory_view)} publications, {inventory_view['matches'].sum():,} match rows")

season,2021/2022,2022/2023,2023/2024,2024/2025
league,,,,
Bundesliga,305,306,306,306
Bundesliga_2,306,306,306,306
Championship,552,552,552,552
Eredivisie,306,306,306,306
La_Liga,380,380,380,380
La_Liga_2,462,462,462,462
Ligue_1,380,380,306,306
Ligue_2,379,379,380,306
Premier_League,380,380,380,380


52 publications, 18,726 match rows


## One real match: identifiers and clocks

The saved example is Brentford versus Arsenal, `event_id=12436410`. `competition_id=17` identifies the Premier League; `season_id=61627` identifies 2024/25. Home and away team IDs are 50 and 42. Names aid reading; IDs anchor joins.

`kickoff_utc` is provider scheduled/revised start in Unix seconds. `event_metadata_observed_at` is when the collector retrieved the metadata. Converting seconds to readable UTC below is presentation only: it does not change either meaning, infer a final-whistle time, or prove historical availability.

In [3]:
match_example = pd.DataFrame(inspection["samples"]["Premier_League_24_25"]["tables"]["matches"]["example"])
example_view = match_example[["event_id", "competition_id", "season_id", "home_id", "away_id", "home_name", "away_name", "round", "status"]].copy()
example_view["scheduled_start_utc"] = pd.to_datetime(match_example["kickoff_utc"], unit="s", utc=True)
example_view["metadata_fetched_utc"] = pd.to_datetime(match_example["event_metadata_observed_at"], unit="s", utc=True)
display(example_view)

,event_id,competition_id,season_id,home_id,away_id,home_name,away_name,round,status,scheduled_start_utc,metadata_fetched_utc
0,12436410,17,61627,50,42,Brentford,Arsenal,19,finished,2025-01-01 17:30:00+00:00,2026-09-13 16:40:19.579999924+00:00


## A statistic row is not a match row

The full observed statistics grain is `(event_id, period, group_name, key, side)`. In Premier League 2024/25, `ALL/totalShotsOnGoal` appears in both `Match overview` and `Shots` groups within `statistics.parquet`: 1,520 stored rows represent 760 event/side pairs. The separate event-level `shots.parquet` contains 9,883 rows. Agreed model boundary: keep the shots table separate from the other tables and let each model select its sources explicitly. Repeated measure names across separate inputs need no deduplication. Preserve table identity and group when selecting statistics; summing these two groups would double-count the measure.

Missingness is also visible: Bundesliga 2024/25 has 306 matches but only 610 `ALL/cornerKicks` observations (two event/side observations are absent). Premier League pregame tables each have only 740 of 760 possible event/side records. These gaps must not turn into zeros or remove matches during loading.

In [4]:
stat_coverage_view = pd.DataFrame([
    {"season": season, "key": key, "stored_rows": detail["rows"],
     "event_side_pairs": detail["unique_event_sides"], "groups": ", ".join(detail["groups"])}
    for season, sample in inspection["samples"].items()
    for key, detail in sample["pilot_stat_coverage"].items()
])
display(stat_coverage_view)

,season,key,stored_rows,event_side_pairs,groups
0,Premier_League_21_22,cornerKicks,760,760,Match overview
1,Premier_League_21_22,totalShotsOnGoal,1520,760,"Match overview, Shots"
2,Premier_League_21_22,shotsOnGoal,760,760,Shots
3,Premier_League_21_22,expectedGoals,0,0,
4,Premier_League_24_25,cornerKicks,760,760,Match overview
5,Premier_League_24_25,totalShotsOnGoal,1520,760,"Match overview, Shots"
6,Premier_League_24_25,shotsOnGoal,760,760,Shots
7,Premier_League_24_25,expectedGoals,1520,760,"Match overview, Shots"
8,Bundesliga_24_25,cornerKicks,610,610,Match overview
9,Bundesliga_24_25,totalShotsOnGoal,1220,610,"Match overview, Shots"


## Optional interactive inspection with D-Tale

[D-Tale](https://github.com/man-group/dtale) 3.22.0 is installed in this kernel's environment. Set `OPEN_DTALE = True` to inspect the 52-row inventory snapshot. `view` embeds the grid; the printed URL also opens it in a browser. This is an in-memory display; source Parquet files are not written. Run the optional shutdown cell when finished.

The setup check verified imports, table creation and HTTP 200 on loopback, then stopped its test server. Full browser controls have not been verified. This optional cell is disabled for a clean automatic notebook run.

In [6]:
OPEN_DTALE = True
if OPEN_DTALE:
    import dtale
    view = dtale.show(inventory_view.copy(), host="127.0.0.1", open_browser=False,
                    notebook=True, allow_cell_edits=False)
    display(view)
    print(view._main_url)

http://127.0.0.1:40000/dtale/main/1


In [7]:
# When finished with an interactive session, set this to True and run the cell.
CLOSE_DTALE = True
if CLOSE_DTALE and "view" in globals():
    view.kill()

2026-09-15 14:45:02,770 - INFO     - Executing shutdown...
2026-09-15 14:45:02,775 - INFO     - Not running with the Werkzeug Server, exiting by searching gc for BaseWSGIServer


## Resolver proposal accepted for this increment

After inspecting the full season, the user chose to proceed with the resolver. The implemented function and its step-by-step walkthrough appear in **Checkpoint 1** below. The next table-reading function remains a separate discussion checkpoint.

## Full season inspection before implementation

The live D-Tale grid below displays **Premier League 2024/25: 380 matches and all 49 source columns**. Scroll horizontally inside the grid to inspect every column. The 14 nested columns are complete JSON text; source files are unchanged.

This cell embeds the existing local season session directly in this notebook. The server runs independently of the kernel, so rerunning the cell reconnects to the same view. It does not open another browser tab. The earlier optional D-Tale example displays only the inventory summary; this is the actual season file.

In [1]:
from pathlib import Path
import json
from IPython.display import IFrame, display

session_file = Path("C:/Users/luisi/Documents/Programming/Python/xDiyo/docs/analytics/dtale_season_session.json")
season_session = json.loads(session_file.read_text(encoding="utf-8"))
display(IFrame(src=season_session["url"], width="100%", height=760))

# Checkpoint 1: resolve one season

The full season grid is useful for exploration, but it is much larger than what this function needs. Think of the resolver as locating a box and reading its packing list: it records which published version and files to use, before opening the Parquet data.

```text
explicit data root + season name
    -> season publication JSON
    -> linked canonical version JSON
    -> ResolvedSeason record
```

There is one new function plus its result record. The notebook imports both; it does not duplicate their implementation.

In [1]:
from pathlib import Path
import inspect
import sys
import pandas as pd
from IPython.display import Code, display
from xdiyo_analytics.data.exports import ResolvedSeason, resolve_season_export

assert Path(sys.executable).resolve() == Path("C:/Users/luisi/Documents/Programming/Python/.misc314/Scripts/python.exe").resolve()
data_root = Path("C:/Users/luisi/Documents/Programming/Python/xDiyo/data/xDiyo_data")
resolved = resolve_season_export(data_root, "Premier_League_24_25")

# Display a small summary instead of printing the entire metadata dictionary.
display(pd.DataFrame([{
    "competition_id": resolved.manifest["scope"]["league_id"],
    "season_id": resolved.manifest["scope"]["season_id"],
    "season_start": resolved.manifest["scope"]["season_start"],
    "season_end": resolved.manifest["scope"]["season_end"],
    "version": resolved.manifest["version"],
    "declared_matches": resolved.manifest["tables"]["matches"]["rows"],
    "declared_tables": len(resolved.manifest["tables"]),
}]))
print("Source module:", inspect.getfile(resolve_season_export))

,competition_id,season_id,season_start,season_end,version,declared_matches,declared_tables
0,17,61627,2024,2025,6d950fa327de4fc29c796c7cca101606,380,15


Source module: C:\Users\luisi\Documents\Programming\Python\xDiyo\src\xdiyo_analytics\data\exports.py


## 1. Arguments and result record

`data_root` is the directory containing the season sidecars. `season_stem` is the filename without an extension, such as `Premier_League_24_25`. It is deliberately explicit: selecting all leagues or automatically taking the latest season would hide an experiment choice.

The `-> ResolvedSeason` annotation documents the return type. It is **one record**, not a table of match observations. Its six metadata fields are:

| Fields | Meaning |
| --- | --- |
| `publication_path`, `manifest_path` | Absolute paths to the two JSON files we read. |
| `publication_sha256`, `manifest_sha256` | Fingerprints of those exact JSON byte sequences. |
| `publication`, `manifest` | Captured parsed metadata, including the linked table descriptors and provenance. |

A dataclass supplies the constructor and named fields. `frozen=True` prevents assigning a different value to a field. **It does not recursively freeze the dictionaries**: callers must treat them as read-only. This keeps the first result easy to inspect, but it is not a deeply immutable object.

`dict[str, Any]` means string keys whose values may be integers, strings, lists, objects or nulls, as JSON permits. We validate the entries needed for resolution, while preserving optional metadata unchanged.

The record also retains `publication_bytes` internally. These exact bytes allow a saved selection to reproduce the original publication fingerprint without rereading a moving pointer.

In [2]:
display(Code(inspect.getsource(ResolvedSeason), language="python"))
from xdiyo_analytics.data.exports import _resolve_season_publication
display(Code(inspect.getsource(resolve_season_export), language='python'))
resolver_source = inspect.getsource(_resolve_season_publication)
print("Signature:", inspect.signature(resolve_season_export))

@dataclass(frozen=True)
class ResolvedSeason:
    """Paths, fingerprints and captured metadata for one published season.

    Table counts and hashes are publisher declarations, not verified table data.
    Frozen fields prevent reassignment; the two metadata dictionaries are still
    mutable Python objects and should be treated as read-only by callers.
    """

    publication_path: Path
    manifest_path: Path
    publication_sha256: str
    manifest_sha256: str
    publication: dict[str, Any]
    manifest: dict[str, Any]
    publication_bytes: bytes = field(repr=False)

def resolve_season_export(data_root: Path, season_stem: str) -> ResolvedSeason:
    """Resolve one schema-2 season sidecar to its explicitly linked version.

    ``season_stem`` is a filename stem such as ``Premier_League_24_25``.
    Read each JSON file once and fingerprint exactly the bytes that were parsed.
    Preserve optional metadata, including any future team-season descriptor.

    Raise FileNotFoundError for absent files, ValueError for invalid metadata or
    selection, and propagate other filesystem errors. No Parquet, CURRENT pointer,
    source writes, normalization or automatic population selection is involved.
    """
    return _resolve_season_publication(data_root, season_stem)

Signature: (data_root: pathlib.Path, season_stem: str) -> xdiyo_analytics.data.exports.ResolvedSeason


## 2. Select and read the publication

The public resolver now delegates these unchanged checks to `_resolve_season_publication`, so a saved publication snapshot can use exactly the same checks. Its leading underscore marks an internal helper, not a function users need to call. This is the actual first section of that helper. The pattern check accepts the inspected `league_YY_YY` naming convention and rejects directory segments or an appended file extension. `Path.resolve()` produces absolute paths; `is_relative_to(root)` keeps the selected file within the chosen source directory.

`read_bytes()` reads the file **once**. `json.loads()` interprets those bytes as structured metadata; later the same bytes produce its fingerprint. Reading once avoids parsing one version and hashing another if a publisher updates the file between operations.

The JSON must be an object, `complete` must literally be `True`, the schema must be `'2'`, and `season_file` must match the requested name. `complete` is a publication declaration, not proof of every fixture or statistic being available. The resolver also works when the named Parquet file is not present yet, because table availability belongs to the next step.

In [3]:
display(Code(resolver_source[:resolver_source.index('    link =')], language="python"))

def _resolve_season_publication(
    data_root: Path, season_stem: str, publication_bytes: bytes | None = None,
) -> ResolvedSeason:
    """Apply the same resolution checks to a live or saved publication snapshot."""
    if not isinstance(season_stem, str) or not re.fullmatch(
        r"[A-Za-z][A-Za-z0-9_]*_[0-9]{2}_[0-9]{2}", season_stem
    ):
        raise ValueError("season_stem must be a league_YY_YY filename stem")

    root = Path(data_root).resolve()
    publication_path = (root / f"{season_stem}.manifest.json").resolve()
    if not publication_path.is_relative_to(root):
        raise ValueError("Publication path is outside data_root")
    if publication_bytes is None:
        publication_bytes = publication_path.read_bytes()
    try:
        publication = json.loads(publication_bytes)
    except (json.JSONDecodeError, UnicodeDecodeError) as exc:
        raise ValueError(f"Invalid JSON in {publication_path}") from exc
    if not isinstance(publication, dict):
        raise ValueError(f"Publication must be a JSON object: {publication_path}")
    if publication.get("complete") is not True:
        raise ValueError(f"Season publication is not complete: {season_stem}")
    if publication.get("schema_version") != "2":
        raise ValueError("Unsupported publication schema_version; expected '2'")
    if publication.get("season_file") != f"{season_stem}.parquet":
        raise ValueError("Publication season_file does not match season_stem")

## 3. Follow the selected version and check identity

`publication['manifest']` is a relative link to the canonical version. The resolver follows **that link once**. There is no scan of the `versions/` directory and no read of `CURRENT.json`.

The canonical scope supplies stable competition and season IDs. `rsplit('_', 2)` separates the two year suffixes while keeping league names such as `Premier_League` intact. `% 100` compares the full stored year (2024) with its filename suffix (24); the manifest, not a century guess, supplies the full year. The adapter currently requires consecutive football season years.

IDs must be positive integers. The exact `type(...) is int` check rejects JSON booleans, because Python otherwise treats `True` as an integer. The linked path must agree with the declared competition ID, season ID and version using the inspected `_tables/competition=.../season=.../versions/.../manifest.json` layout. This is a deliberate schema-2 adapter contract, not a general reader for arbitrary JSON directory layouts.

In [4]:
display(Code(resolver_source[resolver_source.index('    link ='):resolver_source.index('    tables =')], language="python"))
print("Pinned manifest:", resolved.manifest_path)

link = publication.get("manifest")
    if not isinstance(link, str) or not link or Path(link).is_absolute():
        raise ValueError("Publication manifest link must be a nonempty relative path")
    manifest_path = (publication_path.parent / link).resolve()
    if not manifest_path.is_relative_to(root):
        raise ValueError("Canonical manifest path is outside data_root")
    manifest_bytes = manifest_path.read_bytes()
    try:
        manifest = json.loads(manifest_bytes)
    except (json.JSONDecodeError, UnicodeDecodeError) as exc:
        raise ValueError(f"Invalid JSON in {manifest_path}") from exc
    if not isinstance(manifest, dict):
        raise ValueError(f"Canonical manifest must be a JSON object: {manifest_path}")
    if manifest.get("schema_version") != "2":
        raise ValueError("Unsupported canonical schema_version; expected '2'")

    scope = manifest.get("scope")
    if not isinstance(scope, dict):
        raise ValueError("Canonical manifest must contain a scope object")
    for key in ("league_id", "season_id", "season_start", "season_end"):
        if type(scope.get(key)) is not int or scope[key] <= 0:
            raise ValueError(f"scope.{key} must be a positive integer")
    league, start, end = season_stem.rsplit("_", 2)
    if (scope.get("league_name") != league
            or scope["season_start"] % 100 != int(start)
            or scope["season_end"] % 100 != int(end)
            or scope["season_end"] != scope["season_start"] + 1):
        raise ValueError("Canonical scope does not match the requested league/season")

    version = manifest.get("version")
    if not isinstance(version, str) or not re.fullmatch(r"[A-Za-z0-9_-]+", version):
        raise ValueError("Canonical manifest must contain a valid version")
    expected_path = (root / "_tables" / f"competition={scope['league_id']}"
                     / f"season={scope['season_id']}" / "versions" / version
                     / "manifest.json").resolve()
    if manifest_path != expected_path:
        raise ValueError("Manifest path does not match its scope IDs and version")
    if not isinstance(manifest.get("parser_version"), str) or not manifest["parser_version"]:
        raise ValueError("Canonical manifest must contain a parser_version string")

Pinned manifest: C:\Users\luisi\Documents\Programming\Python\xDiyo\data\xDiyo_data\_tables\competition=17\season=61627\versions\6d950fa327de4fc29c796c7cca101606\manifest.json


## 4. Check table declarations without reading tables

`tables` maps each table name to a **descriptor**: its filename, declared number of rows, byte count and expected SHA256 digest. `matches` is required because this is a football season export. Other tables are retained, including a future `team_seasons` table when present.

The loop checks the shape of every descriptor. Filenames must be local `.parquet` names; row and byte counts must be nonnegative integers; each hash must be 64 hexadecimal characters. The two manifests must agree on the declared match count.

These checks make the metadata usable, but they do not establish that table bytes exist or match their hashes. Notice that the function never imports PyArrow or pandas. The small DataFrame below is only a notebook presentation of the descriptors.

In [5]:
display(Code(resolver_source[resolver_source.index('    tables ='):resolver_source.index('    return ResolvedSeason(')], language="python"))
descriptors = pd.DataFrame.from_dict(resolved.manifest["tables"], orient="index")
descriptors.index.name = "table"
display(descriptors[["file", "rows", "bytes"]])
print("team_seasons declared:", "team_seasons" in resolved.manifest["tables"])

tables = manifest.get("tables")
    if not isinstance(tables, dict) or "matches" not in tables:
        raise ValueError("Canonical tables must include a matches descriptor")
    for name, info in tables.items():
        if not isinstance(info, dict):
            raise ValueError(f"Table descriptor must be an object: {name}")
        filename = info.get("file")
        if (not isinstance(filename, str) or not filename.endswith(".parquet")
                or any(char in filename for char in '/\\:') or filename == ".parquet"):
            raise ValueError(f"Table file must be a local Parquet filename: {name}")
        for key in ("rows", "bytes"):
            if type(info.get(key)) is not int or info[key] < 0:
                raise ValueError(f"Table {name}.{key} must be a nonnegative integer")
        if not isinstance(info.get("sha256"), str) or not re.fullmatch(
            r"[0-9a-fA-F]{64}", info["sha256"]
        ):
            raise ValueError(f"Table {name}.sha256 must be a SHA256 hex digest")
    if (type(publication.get("matches")) is not int
            or publication["matches"] != tables["matches"]["rows"]):
        raise ValueError("Publication and canonical matches row declarations disagree")

,file,rows,bytes
table,,,
availability,availability.parquet,3145,142861
coverage,coverage.parquet,6080,128410
goal_actions,goal_actions.parquet,4190,590821
goal_sequences,goal_sequences.parquet,1115,576940
heatmap_points,heatmap_points.parquet,573983,4561161
incidents,incidents.parquet,7647,1421490
lineup_players,lineup_players.parquet,15188,3538923
lineup_teams,lineup_teams.parquet,760,23379
matches,matches.parquet,380,24384


team_seasons declared: False


## 5. Return the captured reference

`hashlib.sha256(...).hexdigest()` produces a 64-character fingerprint of the JSON bytes. The two fingerprints are computed locally; hashes inside table descriptors remain publisher expectations until the reader checks them.

The returned record retains the chosen version even if the public sidecar later points to another version. A new call may resolve the newer version; an existing record does not automatically change. This behavior is verified with a temporary two-version fixture, without changing real exports.

This is **selection pinning**, not a copy of the source files. Files could still be changed or deleted externally; the later reader must check the captured expected hashes. Fingerprints identify content and do not independently authenticate who published it.

In [6]:
display(Code(resolver_source[resolver_source.index('    return ResolvedSeason('):], language="python"))
print("Publication JSON SHA256:", resolved.publication_sha256)
print("Canonical JSON SHA256:", resolved.manifest_sha256)

return ResolvedSeason(
        publication_path=publication_path,
        manifest_path=manifest_path,
        publication_sha256=hashlib.sha256(publication_bytes).hexdigest(),
        manifest_sha256=hashlib.sha256(manifest_bytes).hexdigest(),
        publication=publication,
        manifest=manifest,
        publication_bytes=publication_bytes,
    )

Publication JSON SHA256: 08b302fe765f48ca73a40ccceb6f789e6436aa52f656edc7e81f04dc5fb75faf
Canonical JSON SHA256: 1bae8a6113bd9399a89af5bf589bc1b744b6b7ea7de1c1a4863395b7d69c224f


## 6. Errors and verification

- Missing publication or canonical JSON: `FileNotFoundError`, with the missing path.
- Invalid selection, malformed JSON, unsupported schema, incomplete publication, identity mismatch or unusable descriptors: descriptive `ValueError`.
- Other filesystem failures propagate, rather than being disguised as an absent season.
- Missing optional promotion/relegation context is accepted and stays absent; nothing becomes a fabricated `False` flag.

**Verification:** 39 focused tests passed in `tests/analytics/test_exports.py`. They include resolution without any Parquet files, ignored unreferenced versions/CURRENT, reference stability when a fixture publication advances, later optional table metadata, invalid files and inconsistent identities/descriptors. Pytest reported a nonfatal existing cache-directory warning; no cache repair was attempted. One real Premier League 2024/25 resolution also passed. The editable install preserves the collector CLI and existing package declarations.

The example below produces an intentional invalid-selection error before reading files.

In [7]:
try:
    resolve_season_export(data_root, "../Premier_League_24_25")
except ValueError as error:
    print(type(error).__name__ + ":", error)

ValueError: season_stem must be a league_YY_YY filename stem


## Feedback checkpoint

We now have a reference to **which season version to read**, with its metadata attached. We have not read match observations through the new library yet.

Next, after discussing this code, we will define `read_season_table`: accept this reference plus one table name, verify the actual file against its captured hash, and return a table with missing values preserved. Match-level validation follows as a separate function. League selection remains open for the experiment, but does not block this one-season example.

Pause here for questions or design changes. No claim of understanding is inferred from reading this walkthrough.

**Update:** the user asked to proceed. The one-table reader and its walkthrough follow in Checkpoint 2; match-level validation remains the next discussion.

# Checkpoint 2: read one verified table

Now we use the resolved reference to open **one named table**. The concrete call is:

```python
matches = read_season_table(resolved, "matches")
```

`resolved` carries the selected season version and its metadata. `"matches"` is a key in that manifest's `tables` mapping, not a filename or a new season selection. The return value is a pandas DataFrame containing every physical row and column of that table.

For Premier League 2024/25 this means **380 rows × 35 columns**. These are the scalar match fields; the 14 nested component columns in the 49-column top-level season file live in separate canonical tables. We have not dropped those components: this call explicitly selects only `matches`. Statistics and pregame can be read separately with the same function when needed.

In [8]:
from pathlib import Path
import inspect
import sys
import pandas as pd
from IPython.display import Code, display
from xdiyo_analytics.data.exports import resolve_season_export, read_season_table

assert Path(sys.executable).resolve() == Path("C:/Users/luisi/Documents/Programming/Python/.misc314/Scripts/python.exe").resolve()
data_root = Path("C:/Users/luisi/Documents/Programming/Python/xDiyo/data/xDiyo_data")
resolved = resolve_season_export(data_root, "Premier_League_24_25")
matches = read_season_table(resolved, "matches")
print("Read shape:", matches.shape)
display(matches[["event_id", "competition_id", "season_id", "home_id", "away_id",
                 "home_name", "away_name", "round", "kickoff_utc"]].head(5))
reader_source = inspect.getsource(read_season_table)

Read shape: (380, 35)


,event_id,competition_id,season_id,home_id,away_id,home_name,away_name,round,kickoff_utc
0,12436410,17,61627,50,42,Brentford,Arsenal,19,1735752600.0
1,12436419,17,61627,7,38,Crystal Palace,Chelsea,20,1736002800.0
2,12436434,17,61627,48,40,Everton,Aston Villa,21,1736969400.0
3,12436436,17,61627,50,17,Brentford,Manchester City,21,1736883000.0
4,12436437,17,61627,3,14,Wolverhampton,Nottingham Forest,20,1736193600.0


## 1. Keep the selected version trustworthy

The first section checks the arguments and rereads the **already pinned canonical JSON**, comparing its bytes with the fingerprint recorded by the resolver. It does not reread the public sidecar or CURRENT pointer, so a new publication does not silently change this read's selection.

The next equality check addresses a concrete limitation of our result record: its dictionaries are mutable even though dataclass fields are frozen. If someone edits `resolved.manifest`, the reader rejects the changed snapshot instead of trusting an altered filename or expected hash. This protects ordinary use of resolver-produced references; it is not publisher authentication.

An absent table name raises `KeyError` with the available names. Requesting absent `team_seasons` does not return fabricated flags or an empty table suggesting successful observation. Optional absence can be handled explicitly by a later bundle loader.

In [9]:
display(Code(reader_source[:reader_source.index('    version_dir =')], language="python"))

def read_season_table(resolved: ResolvedSeason, table_name: str) -> "pd.DataFrame":
    """Read one whole table from a resolver-produced reference.

    Check the pinned canonical JSON, then the selected table's byte count,
    SHA256 and row count. Decode the same in-memory bytes that were hashed.
    Return all physical columns and rows, in file order, with Arrow-backed
    pandas dtypes and a fresh RangeIndex. Keep ``resolved`` for provenance.

    An undeclared table raises KeyError, a missing file FileNotFoundError,
    and changed metadata/data or invalid Parquet ValueError. No publication
    pointer is reread. No filtering, joins or football-domain validation occurs.
    Memory use includes the entire compressed file and its decoded table.
    """
    import pandas as pd
    import pyarrow as pa
    import pyarrow.parquet as pq

    if not isinstance(resolved, ResolvedSeason):
        raise TypeError("resolved must be a ResolvedSeason from resolve_season_export")
    if not isinstance(table_name, str) or not table_name:
        raise ValueError("table_name must be a nonempty table name")

    # Recheck this fixed version, not the public sidecar, which may have advanced.
    manifest_bytes = resolved.manifest_path.read_bytes()
    if hashlib.sha256(manifest_bytes).hexdigest() != resolved.manifest_sha256:
        raise ValueError(f"Pinned canonical manifest changed: {resolved.manifest_path}")
    manifest = json.loads(manifest_bytes)
    if manifest != resolved.manifest:
        raise ValueError("Captured manifest metadata was modified after resolution")
    if table_name not in manifest["tables"]:
        available = ", ".join(sorted(manifest["tables"]))
        raise KeyError(f"Table {table_name!r} is not declared; available: {available}")
    info = manifest["tables"][table_name]

## 2. Read, size-check and fingerprint the selected file

The selected descriptor tells us the filename, byte count and expected SHA256. We resolve that filename inside its fixed version directory. The path check also rejects a local filename that resolves outside that directory through a link.

`table_path.read_bytes()` reads the whole compressed file once. We first compare its size, then its SHA256 fingerprint. A size check catches truncation or appended bytes cheaply; a hash check also catches changed content of the same size.

The bytes remain in memory for decoding. Reopening the file after hashing could decode different content if it changed between operations. Using the same buffer avoids that gap.

This first reader handles **one whole table in memory**: compressed bytes and decoded data coexist. It does not yet offer streaming, column projection or row filtering.

In [10]:
display(Code(reader_source[reader_source.index('    version_dir ='):reader_source.index('    # Reading a file buffer')], language="python"))
print("Verified table:", resolved.manifest_path.parent / resolved.manifest["tables"]["matches"]["file"])
print("Expected SHA256:", resolved.manifest["tables"]["matches"]["sha256"])

version_dir = resolved.manifest_path.parent.resolve()
    table_path = (version_dir / info["file"]).resolve()
    if table_path.parent != version_dir:
        raise ValueError(f"Table path is outside its pinned version directory: {table_path}")
    table_bytes = table_path.read_bytes()
    if len(table_bytes) != info["bytes"]:
        raise ValueError(f"Table byte count mismatch: {table_path}")
    if hashlib.sha256(table_bytes).hexdigest() != info["sha256"].lower():
        raise ValueError(f"Table SHA256 mismatch: {table_path}")

Verified table: C:\Users\luisi\Documents\Programming\Python\xDiyo\data\xDiyo_data\_tables\competition=17\season=61627\versions\6d950fa327de4fc29c796c7cca101606\matches.parquet
Expected SHA256: 419a521083156f532fcd7abc5d0e4e5814fd0ae220da7a557149af79292b888e


## 3. Decode the same bytes and check the row count

`pa.BufferReader(table_bytes)` gives PyArrow a file-like view of the verified bytes. `pq.ParquetFile(...).read()` decodes exactly that file. It does not infer additional `competition` or `season` columns from the Hive-style directory names.

The `with` block closes the Parquet reader on success or failure. Arrow decoding failures become a descriptive `ValueError`; missing files and other filesystem errors from `read_bytes()` propagate.

`table.num_rows` is the decoded count, which must match the descriptor. This check concerns file integrity. It does not establish that event IDs are unique, scores are plausible, observations were known before a prediction cutoff, or the entire season was collected. Match-level validation is the next function-sized discussion.

In [11]:
display(Code(reader_source[reader_source.index('    # Reading a file buffer'):reader_source.index('    return table.to_pandas')], language="python"))

# Reading a file buffer also avoids inferring Hive partition columns from paths.
    try:
        with pq.ParquetFile(pa.BufferReader(table_bytes)) as parquet:
            table = parquet.read()
    except pa.ArrowException as exc:
        raise ValueError(f"Invalid or unreadable Parquet table: {table_path}") from exc
    if table.num_rows != info["rows"]:
        raise ValueError(f"Table row count mismatch: {table_path}")

## 4. Return a DataFrame with missing values preserved

`types_mapper=pd.ArrowDtype` asks pandas to retain Arrow-backed column types. An integer column can then contain both exact integers and missing values; it need not become floating point just because one value is missing. Boolean `False` remains distinct from missing. The tests also distinguish numeric zero, a null and a valid floating-point NaN.

`ignore_metadata=True` prevents stored pandas index metadata from hiding a physical column inside an index. The result keeps physical columns in file order and gets a fresh `RangeIndex`. That index is a row position, not a football identifier: use `event_id`, `competition_id`, `season_id` and the team IDs for identity.

This reader leaves UTC Unix seconds as seconds, keeps row order, and preserves stored nulls. It does not sort, filter, fill gaps, normalize standings, join teams, flatten nested data or add flags. Keep `resolved` alongside the DataFrame for provenance; a bare DataFrame is not a complete experiment record.

The table below shows every returned column's type and missing count. Entirely missing extra-time and penalty columns remain present in this example.

In [12]:
display(Code(reader_source[reader_source.index('    return table.to_pandas'):], language="python"))
column_summary = pd.DataFrame({
    "dtype": matches.dtypes.astype(str),
    "missing_rows": matches.isna().sum(),
})
column_summary.index.name = "column"
display(column_summary)
print("Index:", type(matches.index).__name__)
print("Event IDs remain integers:", matches["event_id"].dtype)

return table.to_pandas(types_mapper=pd.ArrowDtype, ignore_metadata=True)

,dtype,missing_rows
column,,
event_id,int64[pyarrow],0
competition_id,int64[pyarrow],0
tournament_id,int64[pyarrow],0
season_id,int64[pyarrow],0
status_code,int64[pyarrow],0
round,int64[pyarrow],0
home_id,int64[pyarrow],0
away_id,int64[pyarrow],0
custom_id,string[pyarrow],0


Index: RangeIndex
Event IDs remain integers: int64[pyarrow]


## Errors and evidence

**59 tests passed** across the resolver and reader (39 existing resolver cases plus 20 reader cases). This run disabled pytest's cache provider so the unrelated cache-directory warning did not affect it.

Reader evidence covers:

- exact large integer IDs, nulls, zero, False, valid NaN, nested values and row/column order;
- optional flags appearing later with unknown values preserved;
- unchanged source bytes and independent returned DataFrames;
- undeclared versus missing tables, and reading only the requested table;
- continuing with the pinned version after the public sidecar advances;
- changed canonical JSON or edited captured metadata;
- changed table size/hash, matching-hash invalid Parquet and a wrong declared row count;
- empty typed tables and physical index columns.

The real example above reads the published Premier League 2024/25 matches table successfully. The deliberate absent-table example below makes the error contract visible without changing any files.

In [13]:
try:
    read_season_table(resolved, "team_seasons")
except KeyError as error:
    print(error.args[0])

Table 'team_seasons' is not declared; available: availability, coverage, goal_actions, goal_sequences, heatmap_points, incidents, lineup_players, lineup_teams, matches, passing_edges, player_statistics, pregame, shots, statistics, timing


## Feedback checkpoint

We now have two separate operations: choose a published version, then read a verified table from it. The notebook's `matches` variable is the actual canonical match DataFrame returned by the library.

Next: discuss the match validator's required columns, identifiers, duplicate policy and treatment of missing timestamps. Keep statistics selection/joins and experiment league selection separate. No validator or feature function has been added in this increment.

Pause here for questions or changes before that next function. Reading this walkthrough is not an assessment of understanding.

**Update:** the user approved the validator and a simple loading call with automatic version/hash handling. Checkpoints 3 and 4 implement that next batch.

## Checkpoint 3: validate match structure

**Purpose:** determine whether match rows have usable identities and season scope. This is a pure check: it returns a `MatchValidationReport` and never edits the DataFrame.

**Inputs:** a match DataFrame and its resolved season reference. **Output:** row count and exact match IDs with missing/invalid kickoff. **Errors:** missing columns, malformed IDs, duplicate match IDs, identical opponents or mismatched scope raise `ValueError`. Wrong input objects raise `TypeError`.

The validator deliberately requires no statistics or promotion flags. Match rows have their own grain; repeated event IDs in a shot table are normal and are not subject to this check.

In [14]:
from dataclasses import asdict
from pathlib import Path
import inspect
import pandas as pd
from IPython.display import Code, display
from xdiyo_analytics.data import MatchValidationReport, validate_matches, load_season_table
from xdiyo_analytics.data.exports import resolve_season_export, read_season_table

project_root = Path('C:/Users/luisi/Documents/Programming/Python/xDiyo')
data_root = project_root / 'data/xDiyo_data'
resolved = resolve_season_export(data_root, 'Premier_League_24_25')
matches_for_validation = read_season_table(resolved, 'matches')
match_report = validate_matches(matches_for_validation, resolved)
display(pd.DataFrame([asdict(match_report)]))
validator_source = inspect.getsource(validate_matches)
display(Code(inspect.getsource(MatchValidationReport), language='python'))

,rows,missing_kickoff_event_ids,invalid_kickoff_event_ids
0,380,(),()


@dataclass(frozen=True)
class MatchValidationReport:
    """Structural success plus timing issues requiring a later eligibility policy."""

    rows: int
    missing_kickoff_event_ids: tuple[int, ...]
    invalid_kickoff_event_ids: tuple[int, ...]

### 1. Required columns and exact identities

`event_id`, `home_id`, `away_id`, `competition_id`, `season_id` and `kickoff_utc` must exist. Column labels must be unique. IDs must be positive integer values: the function does not convert a string or float into an identifier. Python's Boolean values need explicit rejection because they otherwise act like integers.

Error messages include counts and a few **row positions**, which remain useful even if a caller supplies unusual index labels. Optional columns and all source values remain untouched.

In [15]:
display(Code(validator_source[:validator_source.index('    if matches["event_id"].duplicated()')], language='python'))

def validate_matches(
    matches: "pd.DataFrame", resolved: ResolvedSeason,
) -> MatchValidationReport:
    """Reject broken identities/scope; report timing issues without altering rows.

    Require unique column labels, event_id, home_id, away_id, competition_id,
    season_id and kickoff_utc. IDs must be positive integer values (not bools,
    strings or floats), with unique events, distinct opponents and manifest scope.
    Kickoff is numeric Unix seconds: missing values and invalid/nonpositive or
    unrepresentable UTC times are reported separately. No seasonal date cutoff,
    fixture completeness, finished-status or historical availability is inferred.
    Empty tables with required columns are valid. Optional columns stay optional.
    Raise TypeError for wrong inputs and ValueError for structural problems.
    """
    import pandas as pd

    if not isinstance(matches, pd.DataFrame):
        raise TypeError("matches must be a pandas DataFrame")
    if not isinstance(resolved, ResolvedSeason):
        raise TypeError("resolved must be a ResolvedSeason")
    if not matches.columns.is_unique:
        raise ValueError("Match table has repeated column labels")
    id_columns = ("event_id", "home_id", "away_id", "competition_id", "season_id")
    missing = sorted(set((*id_columns, "kickoff_utc")) - set(matches.columns))
    if missing:
        raise ValueError(f"Match table is missing required columns: {', '.join(missing)}")

    errors = []
    for name in id_columns:
        invalid_positions = [
            position for position, value in enumerate(matches[name].tolist())
            if isinstance(value, bool) or not isinstance(value, Integral) or value <= 0
        ]
        if invalid_positions:
            errors.append(f"{name}: {len(invalid_positions)} invalid positive integer IDs "
                          f"at row positions {invalid_positions[:5]}")
    if errors:
        raise ValueError("Invalid match identities: " + "; ".join(errors))

### 2. Match grain and scope

One `event_id` represents one match. Duplicate match records cause a failure; the validator does not decide which record to keep. Opponents must differ, and every row must belong to the competition and season already established by the resolver.

Scope is checked using the manifest's IDs, so the user never types them. This does not assert that the export contains every fixture in the competition.

In [16]:
display(Code(validator_source[validator_source.index('    if matches["event_id"].duplicated()'):validator_source.index('    missing_kickoff =')], language='python'))

if matches["event_id"].duplicated().any():
        errors.append("event_id must be unique; duplicate match rows are not removed")
    if matches["home_id"].eq(matches["away_id"]).any():
        errors.append("home_id and away_id must differ")
    for column, scope_key in (("competition_id", "league_id"), ("season_id", "season_id")):
        expected = resolved.manifest["scope"][scope_key]
        if not matches[column].eq(expected).all():
            errors.append(f"{column} must match manifest scope {expected}")
    if errors:
        raise ValueError("Invalid match structure: " + "; ".join(errors))

### 3. Report timing issues and keep the rows

The stored kickoff uses **Unix seconds in UTC**. Nulls and NaN are reported as missing. Strings, Booleans, nonpositive/nonfinite values and numbers outside the UTC datetime range are invalid. Valid numeric values are checked locally without converting the source column.

No season-date boundary is imposed here; postponed matches can require a separate eligibility policy. A valid scheduled start still does not prove actual kickoff, match completion or historical feature availability. Empty tables with the required columns return a zero-row report.

In [17]:
display(Code(validator_source[validator_source.index('    missing_kickoff ='):], language='python'))

# Synthetic change to an in-memory copy only; source files remain untouched.
timing_example = matches_for_validation.head(2).copy()
timing_example['kickoff_utc'] = pd.Series([None, 'unknown'], dtype=object)
display(pd.DataFrame([asdict(validate_matches(timing_example, resolved))]))
print('Rows retained:', len(timing_example))

missing_kickoff = []
    invalid_kickoff = []
    for event_id, value in zip(matches["event_id"].tolist(), matches["kickoff_utc"].tolist()):
        # Only scalar nulls count as missing; malformed containers are invalid.
        if value is None or value is pd.NA or value is pd.NaT:
            missing_kickoff.append(int(event_id))
            continue
        if isinstance(value, bool) or not isinstance(value, Real):
            invalid_kickoff.append(int(event_id))
            continue
        try:
            numeric_value = float(value)
        except (OverflowError, ValueError):
            invalid_kickoff.append(int(event_id))
            continue
        if math.isnan(numeric_value):
            missing_kickoff.append(int(event_id))
            continue
        if not math.isfinite(numeric_value) or numeric_value <= 0:
            invalid_kickoff.append(int(event_id))
            continue
        try:
            datetime.fromtimestamp(numeric_value, tz=timezone.utc)
        except (OverflowError, OSError, ValueError):
            invalid_kickoff.append(int(event_id))
    return MatchValidationReport(len(matches), tuple(missing_kickoff), tuple(invalid_kickoff))

,rows,missing_kickoff_event_ids,invalid_kickoff_event_ids
0,2,"(12436410,)","(12436419,)"


Rows retained: 2


### 4. See a structural failure

This deliberate duplicate demonstrates the error users would receive. We catch it only to keep the tutorial runnable; the production loader lets the error reach its caller.

In [18]:
duplicate_example = pd.concat([matches_for_validation.head(1)] * 2, ignore_index=True)
try:
    validate_matches(duplicate_example, resolved)
except ValueError as exc:
    print(type(exc).__name__ + ':', exc)

ValueError: Invalid match structure: event_id must be unique; duplicate match rows are not removed


## Checkpoint 4: load with one simple call

`load_season_table(data_root, season_stem, table='matches', record_path=None)` is the user-facing entry point. It returns a **normal pandas DataFrame**. The `*` in the signature makes `table` and `record_path` named arguments, keeping their meaning clear.

Its flow is: resolve the named publication → verify the selected file → validate matches when selected → attach provenance and diagnostics. The lower-level functions remain available for inspection, while routine users need only this call. No version or hash is supplied by the user.

In [19]:
matches = load_season_table(data_root, 'Premier_League_24_25', table='matches')
print('Loaded shape:', matches.shape)
display(matches[['event_id', 'home_name', 'away_name', 'kickoff_utc']].head(5))
display(pd.DataFrame([matches.attrs['xdiyo']['validation']]))
loader_source = inspect.getsource(load_season_table)

Loaded shape: (380, 35)


,event_id,home_name,away_name,kickoff_utc
0,12436410,Brentford,Arsenal,1735752600.0
1,12436419,Crystal Palace,Chelsea,1736002800.0
2,12436434,Everton,Aston Villa,1736969400.0
3,12436436,Brentford,Manchester City,1736883000.0
4,12436437,Wolverhampton,Nottingham Forest,1736193600.0


,status,rows,missing_kickoff_event_ids,invalid_kickoff_event_ids
0,structure_valid,380,(),()


### 1. Resolve, read and validate internally

With no record, each call uses the publication current at that call. With an existing record, it restores that season's saved selection. Matches receive the structural check above. Any missing/invalid kickoff emits a warning and appears in the returned diagnostics, with all rows retained.

Other tables receive file integrity checks. Their domain validation is explicitly marked `not_implemented`; the loader does not claim they passed a match validator.

In [20]:
display(Code(loader_source[:loader_source.index('    frame.attrs["xdiyo"]')], language='python'))

def load_season_table(
    data_root: Path, season_stem: str, *, table: str = "matches",
    record_path: Path | None = None,
) -> "pd.DataFrame":
    """Resolve, verify and load a named table without user-supplied fingerprints.

    Matches also receive structural validation; timing issues emit a warning and
    stay in the data. Other tables receive file integrity checks only, independently.
    Return the original rows/types with source and validation metadata in
    ``frame.attrs['xdiyo']``. Pandas transformations may discard these attributes.

    An optional record_path outside data_root saves this season's publication
    snapshot after a successful load. Reusing it pins the same version for any
    table in that season, even when the public sidecar advances or disappears.
    Missing/changed pinned files fail; there is no fallback to newer data.
    Records contain metadata, not table data, and can accompany a relocated root.
    Without a record, each call selects the publication current at that call.
    Existing records are never overwritten; concurrent first writes may raise
    FileExistsError (retry with the now-existing record). Parent directories are
    created only after successful validation. Source exports are never written.
    """
    root = Path(data_root).resolve()
    path = Path(record_path).resolve() if record_path is not None else None
    if path is not None and path.is_relative_to(root):
        raise ValueError("record_path must be outside data_root to preserve source exports")
    reuse_record = path is not None and path.exists()
    resolved = (_read_season_record(root, season_stem, path) if reuse_record
                else resolve_season_export(root, season_stem))
    frame = read_season_table(resolved, table)

    validation = {"status": "not_implemented", "table": table}
    if table == "matches":
        report = validate_matches(frame, resolved)
        validation = {"status": "structure_valid", **asdict(report)}
        missing_count = len(report.missing_kickoff_event_ids)
        invalid_count = len(report.invalid_kickoff_event_ids)
        if missing_count or invalid_count:
            warnings.warn(
                f"Match timing needs attention: {missing_count} missing and "
                f"{invalid_count} invalid kickoff values; rows retained. "
                "See frame.attrs['xdiyo']['validation'] for event IDs.",
                UserWarning, stacklevel=2,
            )

### 2. Keep the evidence available

`matches.attrs['xdiyo']` contains source provenance and validation results. Its source metadata includes the selected manifest with table fingerprints, scope, parser version, collection provenance and coverage declarations. These details are available for inspection without becoming required arguments.

Pandas transformations may discard attributes. Use a saved selection for persistent replay; attributes alone are not an experiment archive.

In [21]:
display(Code(loader_source[loader_source.index('    frame.attrs["xdiyo"]'):loader_source.index('    if path is not None and not reuse_record:')], language='python'))
print('Source:', matches.attrs['xdiyo']['source']['season_stem'])
print('Validation:', matches.attrs['xdiyo']['validation']['status'])

frame.attrs["xdiyo"] = {
        "source": {
            "season_stem": season_stem, "table": table,
            "manifest_path": str(resolved.manifest_path),
            "publication_sha256": resolved.publication_sha256,
            "manifest_sha256": resolved.manifest_sha256,
            "manifest": resolved.manifest,
        },
        "validation": validation,
    }

Source: Premier_League_24_25
Validation: structure_valid


### 3. Save and reuse a season selection automatically

The only extra input is a readable record filename. The first **successful** load creates it; later calls reuse it, even after the season's public pointer changes. One record covers one season and can be shared by its separate table loads.

The record stores the exact publication bytes (Base64 is a reversible encoding, not encryption) and fingerprints. `_read_season_record` checks the record and routes it through the same resolver checks. Missing or changed pinned files fail explicitly; no newer version is substituted. It creates no copies of table data, so keep the referenced exports available.

Records must be outside the source directory and are never overwritten. For a deliberate refresh, choose a new record filename. This is a season-selection record; a later experiment runner can collect one per selected season alongside model/feature configuration.

In [22]:
selection_path = project_root / 'docs/analytics/loader_demo_selection.json'
pinned_matches = load_season_table(
    data_root, 'Premier_League_24_25', table='matches', record_path=selection_path,
)
replayed_matches = load_season_table(
    data_root, 'Premier_League_24_25', table='matches', record_path=selection_path,
)
pd.testing.assert_frame_equal(pinned_matches, replayed_matches)
assert pinned_matches.attrs == replayed_matches.attrs
print('Saved selection:', selection_path)
print('Replay returned the same data and provenance:', replayed_matches.shape)
display(Code(loader_source[loader_source.index('    if path is not None and not reuse_record:'):], language='python'))

Saved selection: C:\Users\luisi\Documents\Programming\Python\xDiyo\docs\analytics\loader_demo_selection.json
Replay returned the same data and provenance: (380, 35)


if path is not None and not reuse_record:
        record = {
            "schema_version": 1, "season_stem": season_stem,
            "publication_base64": base64.b64encode(resolved.publication_bytes).decode("ascii"),
            "publication_sha256": resolved.publication_sha256,
            "manifest_sha256": resolved.manifest_sha256,
        }
        payload = json.dumps(record, indent=2) + "\n"
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("x", encoding="utf-8") as stream:
            stream.write(payload)
    return frame

### 4. Inspect the internal replay helper

This helper reads a record, checks its schema/season and snapshot fingerprint, applies the shared resolver checks, and verifies the pinned canonical manifest fingerprint. The current public sidecar is not read on replay. Its name starts with `_` because it is an internal implementation detail.

In [23]:
from xdiyo_analytics.data.loading import _read_season_record
display(Code(inspect.getsource(_read_season_record), language='python'))

def _read_season_record(root: Path, season_stem: str, path: Path) -> ResolvedSeason:
    """Restore a snapshot through the ordinary resolver's metadata checks."""
    try:
        record = json.loads(path.read_bytes())
        if (not isinstance(record, dict) or type(record.get("schema_version")) is not int
                or record["schema_version"] != 1 or record.get("season_stem") != season_stem):
            raise ValueError("Record schema or season selection does not match")
        publication_bytes = base64.b64decode(record["publication_base64"], validate=True)
        if hashlib.sha256(publication_bytes).hexdigest() != record["publication_sha256"]:
            raise ValueError("Saved publication snapshot hash mismatch")
        manifest_sha256 = record["manifest_sha256"]
        if not isinstance(manifest_sha256, str) or len(manifest_sha256) != 64:
            raise ValueError("Invalid saved manifest fingerprint")
    except (KeyError, TypeError, ValueError, UnicodeDecodeError, binascii.Error) as exc:
        raise ValueError(f"Invalid season record {path}: {exc}") from exc
    resolved = _resolve_season_publication(root, season_stem, publication_bytes)
    if resolved.manifest_sha256 != manifest_sha256:
        raise ValueError(f"Pinned canonical manifest changed since season record: {path}")
    return resolved

### 5. Load shots separately from the same season selection

The same API returns the shot table independently. There is no implicit join, aggregation or deduplication. Each future model will explicitly select its own tables and statistics groups.

In [24]:
shots = load_season_table(
    data_root, 'Premier_League_24_25', table='shots', record_path=selection_path,
)
display(pd.DataFrame([
    {'table': 'matches', 'rows': len(pinned_matches), 'columns': len(pinned_matches.columns),
     'domain_validation': pinned_matches.attrs['xdiyo']['validation']['status']},
    {'table': 'shots', 'rows': len(shots), 'columns': len(shots.columns),
     'domain_validation': shots.attrs['xdiyo']['validation']['status']},
]))
assert (pinned_matches.attrs['xdiyo']['source']['manifest_sha256']
        == shots.attrs['xdiyo']['source']['manifest_sha256'])
print('Both tables use the same saved season version.')

,table,rows,columns,domain_validation
0,matches,380,35,structure_valid
1,shots,9883,34,not_implemented


Both tables use the same saved season version.


## Verification and feedback checkpoint

**109 focused tests passed** in `.misc314`: existing resolver/reader behavior plus match structure, exact IDs, missing/invalid timing, independent shot rows, automatic provenance, saved selection reuse after pointer advancement/removal, relocated sources and changed/missing-file failures. Failed validation creates no record. Existing records are never overwritten.

The real examples above use the library and preserve source observations. All loader walkthrough sections (Checkpoints 1–4) were executed together in a fresh `misc314_py314` kernel. The earlier user-edited D-Tale/inspection cells were preserved and not rerun, so this is deliberately not a full-notebook execution.

**Pause here for feedback on these two public functions.** The next discussion is which statistics and optional context the first model needs, including selection keys and table-specific validation before joins. Experiment league selection remains open. Multi-season orchestration, model configuration records, features and training remain future steps; reading this walkthrough is not an assessment of understanding.

## Checkpoint 5: inspect a season before choosing tables

**Purpose:** discover available tables and understand their physical columns without decoding observations. `inspect_season(data_root, season_stem, record_path=None)` returns a DataFrame indexed by table name. Versions and fingerprints remain internal.

The result separates a **documented description** of each known table's role from **actual column names and Arrow types read from its file**. Unknown tables remain listed, with a description explaining that their role has not been documented. Descriptions do not guarantee completeness or nonmissing values.

The matching library guide is [Season inspection and standings](../docs/analytics/season_inspection.md).

In [1]:
from pathlib import Path
import inspect
import pandas as pd
from IPython.display import Code, display
from xdiyo_analytics.data import inspect_season, load_season_table

project_root = Path('C:/Users/luisi/Documents/Programming/Python/xDiyo')
data_root = project_root / 'data/xDiyo_data'
selection_path = project_root / 'docs/analytics/loader_demo_selection.json'
catalogue = inspect_season(data_root, 'Premier_League_24_25', record_path=selection_path)
display(catalogue[['declared_rows', 'column_count', 'description']])
inspection_source = inspect.getsource(inspect_season)
print('Available tables:', len(catalogue))

,declared_rows,column_count,description
table,,,
availability,3145,15,"Reported player availability/absence context, ..."
coverage,6080,8,Collection status and latest attempt status fo...
goal_actions,4190,21,"Actions within goal-linked sequences: actors, ..."
goal_sequences,1115,17,"Goal-linked sequences: scorer/team, classifica..."
heatmap_points,573983,11,"Team spatial points with coordinates, weights,..."
incidents,7647,14,"Match incidents: type/class, player, side and ..."
lineup_players,15188,16,"Player lineup entries: identity, playing posit..."
lineup_teams,760,10,"Team lineup context: formation, roster status,..."
matches,380,35,"Match identity, teams, competition, season, sc..."


Available tables: 15


### 1. Use the catalogue in three ways

`catalogue.index` contains the table names. `.loc[table_name, field]` selects one catalogue entry: `columns` returns a Python list; `column_types` returns a name-to-type dictionary; `column_summary` returns a string. `description` explains the table's role in plain language.

`declared_rows` is the manifest's count checked against the Parquet footer, not a count obtained by reading observations. `column_count` and `file_bytes` help you choose which tables to load. Physical column order and nested type descriptions are preserved.

In [2]:
print('Tables:', catalogue.index.tolist())
print('\nPregame columns:', catalogue.loc['pregame', 'columns'])
print('\nPregame types:', catalogue.loc['pregame', 'column_types'])
print('\nPregame summary:', catalogue.loc['pregame', 'column_summary'])
print('\nPregame meaning:', catalogue.loc['pregame', 'description'])

Tables: ['availability', 'coverage', 'goal_actions', 'goal_sequences', 'heatmap_points', 'incidents', 'lineup_players', 'lineup_teams', 'matches', 'passing_edges', 'player_statistics', 'pregame', 'shots', 'statistics', 'timing']

Pregame columns: ['event_id', 'source_key', 'raw_hash', 'observed_at', 'side', 'team_id', 'position', 'value_json']

Pregame types: {'event_id': 'int64', 'source_key': 'string', 'raw_hash': 'string', 'observed_at': 'double', 'side': 'string', 'team_id': 'int64', 'position': 'double', 'value_json': 'string'}

Pregame summary: event_id: int64, source_key: string, raw_hash: string, observed_at: double, side: string, team_id: int64, position: double, value_json: string

Pregame meaning: Match-specific team positions in position, plus provider form/context in value_json; not a complete league standings table.


### 2. Resolve the source or reuse an existing selection

Without `record_path`, the inspector follows the current publication through the existing resolver. With it, the inspector reuses the saved publication and fingerprint checks. Inspection is read-only: it does **not** create a missing record. First create the selection with a successful `load_season_table(..., record_path=...)` call, as demonstrated in Checkpoint 4.

The local imports keep heavier pandas/Arrow dependencies out of metadata-only resolver imports. The source reference fixes the version for every table in this inspection.

In [3]:
display(Code(inspection_source[:inspection_source.index('    entries =')], language='python'))

def inspect_season(
    data_root: Path, season_stem: str, *, record_path: Path | None = None,
) -> "pd.DataFrame":
    """Return a table catalogue indexed by name using Parquet metadata only.

    Columns: description, declared_rows, file_bytes, column_count, columns (list),
    column_types (name/type dict) and column_summary (name/type string).
    Preserve physical column order and nested Arrow types. Descriptions document
    known table roles; unknown tables remain visible with an explicit fallback.

    Resolve the named publication automatically, or reuse an EXISTING selection
    record. Unlike load_season_table, inspection creates no record or directories.
    record_path must be outside data_root. File paths must stay in the pinned
    version directory. Check file size and footer row count against the manifest.
    Missing files raise FileNotFoundError; invalid metadata, paths, sizes, counts
    or unreadable Parquet footers raise ValueError. Other I/O errors propagate.

    Do not decode row values or verify whole-table hashes. A readable footer is
    not proof of valid data pages. Source provenance and this inspection boundary
    are explicit in catalogue.attrs['xdiyo']; use load_season_table for verified
    observation reads. Field nullability is schema metadata, not measured missingness.
    """
    import pandas as pd
    import pyarrow as pa
    import pyarrow.parquet as pq

    root = Path(data_root).resolve()
    path = Path(record_path).resolve() if record_path is not None else None
    if path is not None and path.is_relative_to(root):
        raise ValueError("record_path must be outside data_root to preserve source exports")
    resolved = (_read_season_record(root, season_stem, path) if path is not None
                else resolve_season_export(root, season_stem))
    version_dir = resolved.manifest_path.parent.resolve()

### 3. Inspect each physical file

The loop considers only tables declared by that manifest, in alphabetical order. Each path must remain within the selected version directory. The same open file supplies its byte count and Parquet footer. The footer supplies the schema and row count without decoding rows.

Missing files raise `FileNotFoundError`; invalid metadata, escaped paths, mismatched sizes/counts, unreadable footers or repeated column labels raise `ValueError`. Repeated labels are rejected because a dictionary of column types could otherwise silently lose a column.

`TABLE_DESCRIPTIONS` documents known schema-2 roles based on collector schemas and parsers. It never overrides the columns actually present in the file.

In [4]:
display(Code(inspection_source[inspection_source.index('    entries ='):inspection_source.index('    catalogue =')], language='python'))

entries = []
    for name, descriptor in sorted(resolved.manifest["tables"].items()):
        table_path = (version_dir / descriptor["file"]).resolve()
        if table_path.parent != version_dir:
            raise ValueError(f"Table path is outside its pinned version directory: {table_path}")
        # Size and footer come from the same open file, without decoding data pages.
        with table_path.open("rb") as stream:
            stream.seek(0, 2)
            file_bytes = stream.tell()
            if file_bytes != descriptor["bytes"]:
                raise ValueError(f"Table byte count mismatch: {table_path}")
            stream.seek(0)
            try:
                with pq.ParquetFile(stream) as parquet:
                    schema = parquet.schema_arrow
                    rows = parquet.metadata.num_rows
            except pa.ArrowException as exc:
                raise ValueError(f"Invalid or unreadable Parquet metadata: {table_path}") from exc
        if rows != descriptor["rows"]:
            raise ValueError(f"Table footer row count mismatch: {table_path}")
        if len(set(schema.names)) != len(schema.names):
            raise ValueError(f"Repeated physical column names: {table_path}")
        entries.append({
            "table": name,
            "description": TABLE_DESCRIPTIONS.get(name, "No description documented for this table; inspect its physical columns."),
            "declared_rows": descriptor["rows"],
            "file_bytes": file_bytes,
            "column_count": len(schema),
            "columns": schema.names,
            "column_types": {field.name: str(field.type) for field in schema},
            "column_summary": ", ".join(f"{field.name}: {field.type}" for field in schema),
        })

### 4. Keep the inspection boundary visible

A valid footer is not proof that all data pages are unchanged. The inspector checks file sizes and footer row counts but does not calculate whole-table hashes, measure missingness or validate match observations. The normal loader still performs its full file verification when you request data.

The catalogue retains source metadata and explicit inspection flags in `attrs['xdiyo']`. A schema-only catalogue does not promise that a future read will succeed, especially if files change after inspection.

In [5]:
display(Code(inspection_source[inspection_source.index('    catalogue ='):], language='python'))
display(pd.Series(catalogue.attrs['xdiyo']['inspection'], name='inspection'))

catalogue = pd.DataFrame(entries).set_index("table")
    catalogue.attrs["xdiyo"] = {
        "source": {
            "season_stem": season_stem,
            "manifest_path": str(resolved.manifest_path),
            "publication_sha256": resolved.publication_sha256,
            "manifest_sha256": resolved.manifest_sha256,
            "manifest": resolved.manifest,
        },
        "inspection": {
            "mode": "parquet_metadata", "file_sizes_checked": True,
            "footer_row_counts_checked": True, "table_hashes_verified": False,
            "row_values_read": False,
        },
    }
    return catalogue

mode                         parquet_metadata
file_sizes_checked                       True
footer_row_counts_checked                True
table_hashes_verified                   False
row_values_read                         False
Name: inspection, dtype: object

### 5. Where standings belong

The currently exported standings-related information is **`pregame.position`**: the provider's match-specific position for the home or away team. Its intended row grain is `(event_id, side)`, with `team_id` identifying the team. The collector copies `homeTeam.position`/`awayTeam.position` from the pregame-form payload, and retains the whole corresponding team object in `value_json`.

This is **not a complete league standings table** containing every team's points, games played and goal difference at each round. No separate `standings` table is declared in this saved season. `lineup_players.position` instead means a player's playing position.

Read the pregame table separately using the same record. Provider positions remain raw; normalization and historical availability checks belong to later feature design. `observed_at` records collection time and does not prove that the value was available at an original prediction cutoff.

In [6]:
pregame = load_season_table(
    data_root, 'Premier_League_24_25', table='pregame', record_path=selection_path,
)
display(pregame[['event_id', 'side', 'team_id', 'position']].head(6))
display(pd.DataFrame([{
    'pregame_rows': len(pregame),
    'matches_with_pregame_rows': pregame['event_id'].nunique(),
    'declared_matches': catalogue.loc['matches', 'declared_rows'],
    'missing_positions_in_present_rows': int(pregame['position'].isna().sum()),
    'separate_standings_table': 'standings' in catalogue.index,
}]))
# Show the stored provider object without guessing the meaning of its generic 'value'.
import json
display(json.loads(pregame['value_json'].iloc[0]))

,event_id,side,team_id,position
0,12436410,home,50,12.0
1,12436410,away,42,3.0
2,12436419,home,7,15.0
3,12436419,away,38,4.0
4,12436434,home,48,16.0
5,12436434,away,40,8.0


,pregame_rows,matches_with_pregame_rows,declared_matches,missing_positions_in_present_rows,separate_standings_table
0,740,370,380,0,False


{'avgRating': '6.97',
 'form': ['L', 'W', 'L', 'L', 'D'],
 'position': 12,
 'value': '24'}

### 6. Statistics measures are a separate discovery step

`statistics` has structural columns such as `period`, `group_name`, `key`, `side` and `value`. Measures such as `totalShotsOnGoal` are values of `key`, not separate physical columns. Discovering available measure/group combinations requires reading that table's values; `inspect_season` intentionally stops at the schema.

Keep the `shots` table and each model's selected statistics groups separate, as agreed.

## Verification and feedback checkpoint

**120 focused analytics tests passed**, including 11 new inspection cases. They cover physical schemas without row decoding, source preservation, saved-record reuse without a public pointer, missing records/files, nested and empty unknown tables, invalid footers, size/count inconsistencies and changed pinned manifests. A data-page corruption case demonstrates the stated boundary: schema inspection can succeed while the verified loader rejects the file hash.

Checkpoint 5 was executed alone in a fresh `misc314_py314` kernel; all previous notebook cells and their outputs were preserved. It uses the existing selection record created in Checkpoint 4. This was not a full-notebook/D-Tale rerun.

**Pause for feedback on `inspect_season`.** Next, discuss the first model's actual statistics/context selections and their table-specific validation before joins. Standings reconstruction, normalization, feature construction and experiment league selection remain separate decisions.

## Checkpoint 6: validate team statistics against matches

**Goal:** call `validate_statistics(statistics, matches)` and understand the boundary between a structural error and a reported gap. It returns an immutable `StatisticsValidationReport`; both DataFrames stay unchanged.

This checkpoint stands alone in a fresh `misc314_py314` kernel. It loads only the saved Premier League 2024/25 `matches` and `statistics` tables from `data/xDiyo_data`, using the existing selection record. Individual `shots` remain a separate table. No model inputs, joins or feature selections are made here.

The [statistics validation guide](../docs/analytics/statistics_validation.md) contains the full contract. The [verification record](../docs/analytics/statistics_validation_check.json) records tested source fingerprints and bounded real-data evidence. Every source excerpt below comes from the imported library, not a copied implementation.

In [1]:
from pathlib import Path
from dataclasses import asdict
import hashlib
import inspect
import json
import sys
import pandas as pd
import pyarrow as pa
from IPython.display import Code, display
from xdiyo_analytics.data import StatisticsValidationReport, load_season_table, validate_statistics

cp6_root = Path('C:/Users/luisi/Documents/Programming/Python/xDiyo')
cp6_record = cp6_root / 'docs/analytics/loader_demo_selection.json'
cp6_evidence = json.loads((cp6_root / 'docs/analytics/statistics_validation_check.json').read_text())
cp6_expected_source = cp6_evidence['source_sha256_after_real_check']
cp6_source_before = {name: hashlib.sha256((cp6_root / name).read_bytes()).hexdigest()
                     for name in cp6_expected_source}
assert cp6_source_before == cp6_expected_source, 'Source differs from the verified state'

cp6_matches = load_season_table(cp6_root / 'data/xDiyo_data', 'Premier_League_24_25', record_path=cp6_record)
cp6_statistics = load_season_table(cp6_root / 'data/xDiyo_data', 'Premier_League_24_25',
                                 table='statistics', record_path=cp6_record)
cp6_matches_before = cp6_matches.copy(deep=True)
cp6_statistics_before = cp6_statistics.copy(deep=True)
assert (cp6_matches.attrs['xdiyo']['source']['manifest_sha256']
        == cp6_statistics.attrs['xdiyo']['source']['manifest_sha256'])
cp6_function_source = inspect.getsource(validate_statistics)
print('Kernel interpreter:', sys.executable)
display(pd.DataFrame({'table': ['matches', 'statistics'],
                      'rows': [len(cp6_matches), len(cp6_statistics)],
                      'columns': [len(cp6_matches.columns), len(cp6_statistics.columns)]}))

Kernel interpreter: C:\Users\luisi\Documents\Programming\Python\.misc314\Scripts\python.exe


,table,rows,columns
0,matches,380,35
1,statistics,98224,18


### 1. Inputs, output and the real result

`statistics` needs `event_id`, `period`, `group_name`, `key`, `side` and `value`. `matches` needs `event_id`, `home_id` and `away_id`. Optional `team_id` values are checked when present. Extra columns such as `display` remain as supplied.

The report counts statistic rows and **distinct referenced matches**, records the presence of the `team_id` column, and stores three tuples of gaps. Missing row positions are **zero-based input positions**, independent of index labels. Missing sides are `(event_id, 'home'/'away')` pairs in reference-match row order, home before away within each match. The frozen dataclass prevents field reassignment.

The loaded real tables contain **98,224 statistics rows and 380 reference matches**. This selected export has no missing team IDs, no missing values, and no match side without any statistic rows. This does not establish completeness of every statistic in every period.

In [2]:
display(Code(inspect.getsource(StatisticsValidationReport), language='python'))
cp6_report = validate_statistics(cp6_statistics, cp6_matches)
display(pd.Series(asdict(cp6_report), name='real_statistics_report'))
print('Loader metadata still says:', cp6_statistics.attrs['xdiyo']['validation'])

@dataclass(frozen=True)
class StatisticsValidationReport:
    """Structural success and gaps; row positions refer to the input's file order.

    Missing numeric values do not imply an unavailable measure: display may hold
    text. Missing match sides mean no statistic rows at all, not missing coverage
    for a particular measure/period. A later selector defines required measures.
    """

    rows: int
    matches_referenced: int
    team_id_column_present: bool
    missing_team_id_row_positions: tuple[int, ...]
    missing_value_row_positions: tuple[int, ...]
    match_sides_without_statistics: tuple[tuple[int, str], ...]

rows                              98224
matches_referenced                  380
team_id_column_present             True
missing_team_id_row_positions        ()
missing_value_row_positions          ()
match_sides_without_statistics       ()
Name: real_statistics_report, dtype: object

Loader metadata still says: {'status': 'not_implemented', 'table': 'statistics'}


### 2. Required columns and exact integer identities

The first part checks argument types, unique column labels and required fields. It then validates all reference match IDs/teams and statistic event IDs: each must be a positive integer value. Booleans, numeric strings, floats (even `7.0`), nulls and negative/zero IDs fail. The reference table also needs one row per event and different home/away teams.

`TypeError` means an argument is not a DataFrame; `ValueError` means an input violates the structural contract. Error positions refer to row order and usually show only the first five affected positions. This validator checks the reference **identity subset**; it does not repeat match scope or kickoff validation.

In [3]:
cp6_label_start = cp6_function_source.index('    for column in ("period", "group_name", "key"):')
cp6_reference_start = cp6_function_source.index('    # Tuples retain exact integer IDs')
cp6_gap_start = cp6_function_source.index('    missing_values =')
display(Code(cp6_function_source[:cp6_label_start], language='python'))

def validate_statistics(
    statistics: "pd.DataFrame", matches: "pd.DataFrame",
) -> StatisticsValidationReport:
    """Check one statistics table against its season's match identities.

    Require event_id, period, group_name, key, side and value in statistics, and
    event_id/home_id/away_id in matches. IDs must be positive integer values,
    excluding bools/floats/strings. Match IDs must be unique, with distinct teams.
    Statistic labels must be nonempty strings; side must be home or away. Reject
    orphan references and repeated (event_id, period, group_name, key, side).
    The same measure in different groups is allowed. Preserve key casing.

    If team_id is supplied, require it to match the referenced match's side.
    Missing team IDs (including an absent column) are reported, not invented.
    Report scalar null/NaN values without treating missing numeric values as a
    structural error; display/text and all other columns remain untouched.
    No numeric range, per-measure completeness or historical timing policy is
    imposed. Report match sides with no statistic rows at all. Empty statistics
    with required columns are valid. Duplicate column labels are rejected.

    Return an immutable report; never mutate inputs, join/aggregate/filter rows,
    or read files. TypeError denotes wrong argument types; ValueError denotes a
    structural failure. Normally pass matches from load_season_table and use one
    saved season selection for both inputs. This function rechecks the identity
    subset, not match scope/timing, file integrity or version provenance.
    """
    import pandas as pd

    grain = ("event_id", "period", "group_name", "key", "side")
    for label, frame, required in (
        ("statistics", statistics, (*grain, "value")),
        ("matches", matches, ("event_id", "home_id", "away_id")),
    ):
        if not isinstance(frame, pd.DataFrame):
            raise TypeError(f"{label} must be a pandas DataFrame")
        if not frame.columns.is_unique:
            raise ValueError(f"{label} has repeated column labels")
        missing = sorted(set(required) - set(frame.columns))
        if missing:
            raise ValueError(f"{label} is missing required columns: {', '.join(missing)}")

    for label, frame, columns in (
        ("matches", matches, ("event_id", "home_id", "away_id")),
        ("statistics", statistics, ("event_id",)),
    ):
        for column in columns:
            bad = [i for i, value in enumerate(frame[column].tolist())
                if isinstance(value, bool) or not isinstance(value, Integral) or value <= 0]
            if bad:
                raise ValueError(f"{label}.{column}: {len(bad)} invalid positive integer IDs "
                                f"at row positions {bad[:5]}")
    if matches["event_id"].duplicated().any():
        raise ValueError("matches.event_id must be unique")
    if matches["home_id"].eq(matches["away_id"]).any():
        raise ValueError("Match home_id and away_id must differ")

### 3. Labels, exact sides and the complete row identity

`period`, `group_name` and `key` must be strings containing at least one non-whitespace character. The function checks `.strip()` but keeps the original labels: it does not trim or change casing. `side` must be exactly `home` or `away`; `HOME` and `away ` fail.

The full grain (the fields that identify one row) is **`(event_id, period, group_name, key, side)`**. Two rows at that full grain fail even when their values differ. Repeating a key across groups, periods, sides or matches is allowed. This prevents later code from silently collapsing distinct provider entries.

In [4]:
display(Code(cp6_function_source[cp6_label_start:cp6_reference_start], language='python'))

for column in ("period", "group_name", "key"):
        bad = [i for i, value in enumerate(statistics[column].tolist())
            if not isinstance(value, str) or not value.strip()]
        if bad:
            raise ValueError(f"statistics.{column} requires nonempty strings; "
                            f"invalid row positions {bad[:5]}")
    bad_sides = [i for i, value in enumerate(statistics["side"].tolist())
                if not isinstance(value, str) or value not in ("home", "away")]
    if bad_sides:
        raise ValueError(f"statistics.side must be home or away; row positions {bad_sides[:5]}")
    duplicate_positions = [i for i, duplicate in enumerate(
        statistics.duplicated(subset=list(grain), keep=False).tolist()) if duplicate]
    if duplicate_positions:
        raise ValueError("Repeated statistics entries at (event_id, period, group_name, key, side); "
                        f"row positions {duplicate_positions[:5]}")

### 4. See repeated groups in the real data

This saved season has three periods (`ALL`, `1ST`, `2ND`), seven groups and 50 distinct statistic keys. Four keys occur in more than one group. For example, the same match's `totalShotsOnGoal` occurs in both `Match overview` and `Shots`.

The following narrow view displays those separate rows. Dropping `group_name` would make distinct entries appear duplicated, and summing across the groups could count the same provider measure twice. A later model selector must choose its period, group and key deliberately. The `Shots` **statistics group** is also distinct from the individual-event `shots` **table**.

In [5]:
cp6_grain = ['event_id', 'period', 'group_name', 'key', 'side']
cp6_first_event = cp6_statistics['event_id'].iloc[0]
cp6_group_example = cp6_statistics.loc[
    cp6_statistics['event_id'].eq(cp6_first_event)
    & cp6_statistics['period'].eq('ALL')
    & cp6_statistics['key'].eq('totalShotsOnGoal'),
    cp6_grain + ['value', 'display'],
]
display(cp6_group_example)
cp6_key_groups = (cp6_statistics[['key', 'group_name']].drop_duplicates()
                  .groupby('key')['group_name'].nunique())
display(cp6_key_groups[cp6_key_groups.gt(1)].rename('distinct_groups').to_frame())

,event_id,period,group_name,key,side,value,display
8,12436410,ALL,Match overview,totalShotsOnGoal,home,5.0,5
9,12436410,ALL,Match overview,totalShotsOnGoal,away,14.0,14
26,12436410,ALL,Shots,totalShotsOnGoal,home,5.0,5
27,12436410,ALL,Shots,totalShotsOnGoal,away,14.0,14


,distinct_groups
key,
expectedGoals,2
goalkeeperSaves,2
totalShotsOnGoal,2
totalTackle,2


### 5. Match references and supplied team IDs

The dictionary maps each exact event ID to its home and away teams. `itertuples()` avoids the integer-to-float coercion that mixed numeric rows can cause with `iterrows()`. Every statistic must reference a known match; an orphan raises an error before team checks.

When a nonmissing `team_id` is supplied, it must be a positive integer and agree with that match's stated side. Missing team IDs are reported and stay unknown. If the column is absent, every row position is listed as missing and `team_id_column_present` is false. The function never creates or fills that column.

In [6]:
display(Code(cp6_function_source[cp6_reference_start:cp6_gap_start], language='python'))

# Tuples retain exact integer IDs; iterrows can coerce mixed numeric rows.
    match_teams = {int(event): {"home": int(home), "away": int(away)}
                for event, home, away in matches[["event_id", "home_id", "away_id"]]
                .itertuples(index=False, name=None)}
    events = statistics["event_id"].tolist()
    sides = statistics["side"].tolist()
    orphans = [i for i, event in enumerate(events) if event not in match_teams]
    if orphans:
        raise ValueError(f"Statistics reference unknown match IDs at row positions {orphans[:5]}")

    has_team_id = "team_id" in statistics.columns
    team_ids = statistics["team_id"].tolist() if has_team_id else [None] * len(statistics)
    missing_team_ids = []
    for i, (event, side, team) in enumerate(zip(events, sides, team_ids)):
        if pd.api.types.is_scalar(team) and pd.isna(team):
            missing_team_ids.append(i)
        elif isinstance(team, bool) or not isinstance(team, Integral) or team <= 0:
            raise ValueError(f"statistics.team_id must be a positive integer when supplied; row position {i}")
        elif team != match_teams[event][side]:
            raise ValueError(f"statistics.team_id does not match the match's {side} team at row position {i}")

### 6. Report gaps without changing observations

Scalar null/NaN `value` entries become missing-value positions, without raising an error. Arrow nulls and valid Arrow NaNs both qualify. A numeric value may be absent while `display` still contains a percentage, ratio or other text. The validator does not parse that text or substitute it into `value`.

The presence of **any** statistic row covers a match side for this report, even if its value is missing. No row for one particular measure or period is a different question, which the later selector must answer. This function also has no numeric-type or range policy: zero, negative values, infinity and nonmissing text are not rejected by this structural check.

In [7]:
display(Code(cp6_function_source[cp6_gap_start:], language='python'))

missing_values = tuple(i for i, value in enumerate(statistics["value"].tolist())
                        if pd.api.types.is_scalar(value) and pd.isna(value))
    present_sides = set(zip(events, sides))
    absent_sides = tuple((event, side) for event in match_teams for side in ("home", "away")
                        if (event, side) not in present_sides)
    return StatisticsValidationReport(
        rows=len(statistics), matches_referenced=len(set(events)),
        team_id_column_present=has_team_id,
        missing_team_id_row_positions=tuple(missing_team_ids),
        missing_value_row_positions=missing_values,
        match_sides_without_statistics=absent_sides,
    )

### 7. Synthetic gaps, exact large IDs and arbitrary indices

The real selection has no reported gaps, so this synthetic example makes them visible. It uses IDs above `2**53`, where floats can lose integer precision, an Arrow null and a valid Arrow NaN, and repeated string index labels. Those labels do not change the report's row positions.

Expected result: missing team IDs at position `(1,)`, missing values at `(0, 1)`, and both sides of event `7` without statistics. The text remains available in `display`, and the original nullable Arrow dtypes remain intact.

In [8]:
cp6_large_event, cp6_large_home, cp6_large_away = 2**53 + 1, 2**53 + 3, 2**53 + 5
cp6_demo_matches = pa.table({
    'event_id': [cp6_large_event, 7], 'home_id': [cp6_large_home, 42],
    'away_id': [cp6_large_away, 50],
}).to_pandas(types_mapper=pd.ArrowDtype)
cp6_demo_statistics = pa.table({
    'event_id': [cp6_large_event, cp6_large_event], 'period': ['ALL', 'ALL'],
    'group_name': ['Synthetic', 'Synthetic'], 'key': ['ratioExample', 'ratioExample'],
    'side': ['home', 'away'], 'team_id': pa.array([cp6_large_home, None], type=pa.int64()),
    'value': pa.array([None, float('nan')], type=pa.float64(), from_pandas=False),
    'display': ['7/10 (70%)', 'not supplied'],
}).to_pandas(types_mapper=pd.ArrowDtype)
cp6_demo_statistics.index = pd.Index(['same label', 'same label'], name='user_index')
cp6_demo_before = cp6_demo_statistics.copy(deep=True)
cp6_demo_report = validate_statistics(cp6_demo_statistics, cp6_demo_matches)
assert cp6_demo_report.missing_team_id_row_positions == (1,)
assert cp6_demo_report.missing_value_row_positions == (0, 1)
assert cp6_demo_report.match_sides_without_statistics == ((7, 'home'), (7, 'away'))
print('Arrow null count:', cp6_demo_statistics['value'].array.__arrow_array__().null_count)
display(cp6_demo_statistics)
display(pd.Series(asdict(cp6_demo_report), name='synthetic_gap_report'))
cp6_no_teams = cp6_demo_statistics.drop(columns='team_id')
print('Absent team column:', asdict(validate_statistics(cp6_no_teams, cp6_demo_matches)))

Arrow null count: 1


,event_id,period,group_name,key,side,team_id,value,display
user_index,,,,,,,,
same label,9007199254740993,ALL,Synthetic,ratioExample,home,9007199254740995,<NA>,7/10 (70%)
same label,9007199254740993,ALL,Synthetic,ratioExample,away,<NA>,NaN,not supplied


rows                                                   2
matches_referenced                                     1
team_id_column_present                              True
missing_team_id_row_positions                       (1,)
missing_value_row_positions                       (0, 1)
match_sides_without_statistics    ((7, home), (7, away))
Name: synthetic_gap_report, dtype: object

Absent team column: {'rows': 2, 'matches_referenced': 1, 'team_id_column_present': False, 'missing_team_id_row_positions': (0, 1), 'missing_value_row_positions': (0, 1), 'match_sides_without_statistics': ((7, 'home'), (7, 'away'))}


### 8. Read the actual error messages

These independent, intentionally malformed copies show duplicate full grains, invalid sides, orphan event references, disagreement with a supplied team, and a missing required column. Each expected `ValueError` is caught for display; no notebook error output is left behind and the original demonstration table is preserved.

Fix a structural failure by investigating the source/selection or the transformation that caused it. An arbitrary deduplication, guessed ID or filled value is not part of this function's contract.

In [9]:
cp6_error_inputs = {
    'duplicate full grain': pd.concat([cp6_demo_statistics, cp6_demo_statistics.iloc[[0]]]),
    'invalid side': cp6_demo_statistics.assign(side=['HOME', 'away']),
    'orphan reference': cp6_demo_statistics.assign(event_id=[8, 8]),
    'wrong team for side': cp6_demo_statistics.assign(
        team_id=pd.array([cp6_large_away, cp6_large_away], dtype='int64[pyarrow]')),
    'missing required column': cp6_demo_statistics.drop(columns='period'),
}
cp6_error_rows = []
for cp6_label, cp6_bad_input in cp6_error_inputs.items():
    try:
        validate_statistics(cp6_bad_input, cp6_demo_matches)
    except ValueError as error:
        cp6_error_rows.append({'case': cp6_label, 'error': str(error)})
    else:
        raise AssertionError(f'Expected structural error: {cp6_label}')
with pd.option_context('display.max_colwidth', None):
    display(pd.DataFrame(cp6_error_rows))

,case,error
0,duplicate full grain,"Repeated statistics entries at (event_id, period, group_name, key, side); row positions [0, 2]"
1,invalid side,statistics.side must be home or away; row positions [0]
2,orphan reference,"Statistics reference unknown match IDs at row positions [0, 1]"
3,wrong team for side,statistics.team_id does not match the match's home team at row position 0
4,missing required column,statistics is missing required columns: period


### 9. Empty tables and the coverage boundary

An empty statistics DataFrame is valid when it retains the required columns. It reports zero statistic rows and every supplied match side as uncovered. Empty statistics with an empty, correctly shaped match reference table produce no missing sides. Required columns and reference identities are still checked even when there are no statistic rows.

Conversely, one synthetic measure per side is enough to remove the side-coverage gap, even when both values are missing. That success says nothing about a model's required corners, shots, periods or groups.

In [10]:
cp6_empty_report = validate_statistics(cp6_demo_statistics.iloc[:0], cp6_demo_matches)
display(pd.Series(asdict(cp6_empty_report), name='empty_statistics_report'))
cp6_both_empty = validate_statistics(cp6_demo_statistics.iloc[:0], cp6_demo_matches.iloc[:0])
assert cp6_both_empty.rows == 0 and cp6_both_empty.match_sides_without_statistics == ()
cp6_one_match_report = validate_statistics(cp6_demo_statistics, cp6_demo_matches.iloc[:1])
print('Sides without rows:', cp6_one_match_report.match_sides_without_statistics)
print('Missing values still present:', cp6_one_match_report.missing_value_row_positions)

rows                                                                              0
matches_referenced                                                                0
team_id_column_present                                                         True
missing_team_id_row_positions                                                    ()
missing_value_row_positions                                                      ()
match_sides_without_statistics    ((9007199254740993, home), (9007199254740993, ...
Name: empty_statistics_report, dtype: object

Sides without rows: ()
Missing values still present: (0, 1)


### 10. Check preservation and the tested source state

The checks below compare real and synthetic inputs with their saved copies, including source attributes, indices and nullable dtypes. They also confirm that the source fingerprints match the state used by the full analytics suite and bounded real-data check.

`validate_statistics` performs no file I/O or provenance/version comparison. Loading both tables with the same saved selection provides that context separately. Its returned report is not automatically written into the loader's attributes; keep the explicit report alongside the selected input metadata.

In [11]:
pd.testing.assert_frame_equal(cp6_statistics, cp6_statistics_before)
pd.testing.assert_frame_equal(cp6_matches, cp6_matches_before)
pd.testing.assert_frame_equal(cp6_demo_statistics, cp6_demo_before)
assert cp6_statistics.attrs == cp6_statistics_before.attrs
assert cp6_matches.attrs == cp6_matches_before.attrs
cp6_source_after = {name: hashlib.sha256((cp6_root / name).read_bytes()).hexdigest()
                    for name in cp6_expected_source}
assert cp6_source_before == cp6_source_after == cp6_expected_source
print('Real and synthetic inputs preserved; source fingerprints match the checked state.')
for name, digest in cp6_source_after.items():
    if name.endswith(('statistics.py', 'data/__init__.py')):
        print(f'{name} SHA256: {digest}')

Real and synthetic inputs preserved; source fingerprints match the checked state.
src/xdiyo_analytics/data/__init__.py SHA256: 054ea336427856a9471b18af3f42aa0c867f7c407fa1821c2f58aae60bde95e1
src/xdiyo_analytics/data/statistics.py SHA256: 4a13fda9ff4ee840af7c32afcade033ec48d6e15dcfb7af759893fc8b51180bf


## Verification and next discussion

**257 analytics tests passed, including 137 new statistics cases.** They cover exact large and unsigned Arrow IDs, required/repeated columns, malformed labels/IDs/sides, duplicate full grains, permitted repeated keys, orphan references, malformed match identities, team agreement/missingness, null/NaN and text, empty inputs, side coverage and input/index preservation.

The saved Premier League 2024/25 check passed for **98,224 × 18 statistics** against **380 × 35 matches**, with zero reported gaps. Selected input files and the selection record were unchanged. A source fingerprint guard initially detected an edit after the baseline capture; the suite was rerun with matching fingerprints immediately before and after. The checked `statistics.py` SHA256 is `4a13fda9ff4ee840af7c32afcade033ec48d6e15dcfb7af759893fc8b51180bf`.

**Only these 23 new cells (11 code cells) were executed in a fresh `misc314_py314` kernel.** All 79 prior cells and outputs were preserved; this was not a full-notebook rerun and the existing D-Tale cells were not executed. The complete notebook now has 102 cells.

**Next discussion:** select the first model's periods, groups, measures and optional context, then define the needed completeness and timing policies before joins. Structural success here does not establish numeric validity, provider-wide fixture/measure completeness, original pre-match availability or a selected experiment population.

## Checkpoint 7: load separate tables from one selected season

**Goal:** use `load_season(data_root, season_stem, tables=("matches",), record_path=None)` to obtain a `SeasonData` bundle. You choose the table names; the loader resolves one version, reads each requested table against it, runs available validators, and returns the tables separately with provenance and reports.

This independent checkpoint uses the existing Premier League 2024/25 selection record and `data/xDiyo_data`. It requests **matches, statistics and pregame**. Shots are unrequested. The [season bundle guide](../docs/analytics/season_loading.md) explains the API; the [verification record](../docs/analytics/season_loading_check.json) identifies this batch's checked source state.

**Source history:** `season.py` is new, and both loading APIs now share `_write_season_record` from `loading.py`. Earlier source displays and outputs are preserved as historical checkpoint evidence. Checkpoint 6's old batch-level fingerprint guard includes files intentionally changed here and would flag those changes if rerun unchanged. This checkpoint uses a fresh baseline; only Checkpoint 7 is executed in this batch.

In [1]:
from pathlib import Path
from copy import deepcopy
from dataclasses import FrozenInstanceError
import hashlib
import inspect
import json
import sys
import pandas as pd
from IPython.display import Code, display
from xdiyo_analytics.data import SeasonData, load_season, load_season_table
from xdiyo_analytics.data.loading import _write_season_record

cp7_root = Path('C:/Users/luisi/Documents/Programming/Python/xDiyo')
cp7_data_root = cp7_root / 'data/xDiyo_data'
cp7_record = cp7_root / 'docs/analytics/loader_demo_selection.json'
cp7_evidence = json.loads((cp7_root / 'docs/analytics/season_loading_check.json').read_text())
cp7_expected_source = cp7_evidence['source_sha256_after_real_check']
cp7_source_before = {name: hashlib.sha256((cp7_root / name).read_bytes()).hexdigest()
                     for name in cp7_expected_source}
assert cp7_source_before == cp7_expected_source, 'Source differs from the checked bundle state'
cp7_input_before = {name: hashlib.sha256((cp7_root / name).read_bytes()).hexdigest()
                    for name in cp7_evidence['input_sha256_after']}

cp7_season = load_season(cp7_data_root, 'Premier_League_24_25',
                       tables=('matches', 'statistics', 'pregame'), record_path=cp7_record)
cp7_frames_before = {name: frame.copy(deep=True) for name, frame in cp7_season.tables.items()}
cp7_provenance_before = deepcopy(cp7_season.provenance)
cp7_validation_before = deepcopy(cp7_season.validation)
cp7_load_source = inspect.getsource(load_season)
print('Kernel interpreter:', sys.executable)
display(pd.DataFrame([{'table': name, 'rows': len(frame), 'columns': len(frame.columns),
                       'domain_validation': cp7_season.validation[name]['status']}
                      for name, frame in cp7_season.tables.items()]))

Kernel interpreter: C:\Users\luisi\Documents\Programming\Python\.misc314\Scripts\python.exe


,table,rows,columns,domain_validation
0,matches,380,35,structure_valid
1,statistics,98224,18,structure_valid
2,pregame,740,8,not_implemented


### 1. Understand `SeasonData` and its properties

`tables` is a dictionary containing exactly the requested table names in request order. `provenance` describes their shared selection; `validation` maps each selected table to its diagnostic report. The convenient `.matches`, `.statistics`, `.pregame` and `.shots` properties reference those same DataFrames.

Here `.shots` is `None` because shots were **not requested**. Requesting an undeclared table instead raises `KeyError` before reading any table files. A declared but missing file raises `FileNotFoundError`; a requested table that legitimately has zero rows is still an empty DataFrame. The property is not a missing-data fallback.

The dataclass freezes field reassignment. Its dictionaries and DataFrames remain mutable; reports describe the state at loading time. We demonstrate that boundary below.

In [2]:
display(Code(inspect.getsource(SeasonData), language='python'))
print('Requested names:', tuple(cp7_season.tables))
print('matches property is the table:', cp7_season.matches is cp7_season.tables['matches'])
print('statistics property is the table:', cp7_season.statistics is cp7_season.tables['statistics'])
print('Unrequested shots property:', cp7_season.shots)
display(cp7_season.matches[['event_id', 'home_id', 'away_id']].head(3))

@dataclass(frozen=True)
class SeasonData:
    """Separate DataFrames with source provenance and per-table validation reports.

    ``tables`` contains exactly the requested names, in request order. Convenience
    properties return None for tables not requested; an unavailable requested
    table instead makes loading fail. Access any other table through ``tables``.
    Frozen fields prevent reassignment, not edits to nested dictionaries/frames.
    Reports describe the loaded state and are not refreshed after caller edits.
    """

    tables: dict[str, "pd.DataFrame"]
    provenance: dict[str, Any]
    validation: dict[str, dict[str, Any]]

    @property
    def matches(self) -> "pd.DataFrame | None":
        return self.tables.get("matches")

    @property
    def statistics(self) -> "pd.DataFrame | None":
        return self.tables.get("statistics")

    @property
    def pregame(self) -> "pd.DataFrame | None":
        return self.tables.get("pregame")

    @property
    def shots(self) -> "pd.DataFrame | None":
        return self.tables.get("shots")

Requested names: ('matches', 'statistics', 'pregame')
matches property is the table: True
statistics property is the table: True
Unrequested shots property: None


,event_id,home_id,away_id
0,12436410,50,42
1,12436419,7,38
2,12436434,48,40


### 2. Validate arguments, resolve once and precheck availability

`tables` must be a nonempty ordered sequence, such as a list or tuple, containing distinct nonempty strings. A bare string, set or generator raises `TypeError`; empty selections, invalid names or duplicates raise `ValueError`. Names retain their spelling and case. Selecting `statistics` requires explicitly including `matches`, in either order. No reference table is added silently.

The source is resolved **once**. With an existing `record_path`, the saved publication is replayed; otherwise the current publication is selected once. The record must be outside `data_root`. All requested names are checked against that manifest before any table file is read.

Every later read receives this same `ResolvedSeason`. A public pointer changing between reads cannot redirect one table to another version. The existing reader still checks the pinned canonical manifest and each selected file; changed pinned files fail instead of triggering a refresh. This provides consistent selection, not a lock on source files.

In [3]:
cp7_reads_start = cp7_load_source.index('    frames =')
cp7_provenance_start = cp7_load_source.index('    provenance =')
display(Code(cp7_load_source[:cp7_reads_start], language='python'))

def load_season(
    data_root: Path, season_stem: str, *, tables: Sequence[str] = ("matches",),
    record_path: Path | None = None,
) -> SeasonData:
    """Load requested tables from one version, with available domain validation.

    Resolve once, then verify every requested table against that same canonical
    manifest. Require a nonempty ordered sequence of distinct nonempty names.
    Selecting statistics requires explicitly selecting matches as its reference.
    No additional tables are loaded implicitly. Run validate_matches when matches
    is selected and validate_statistics when statistics is selected. Other tables
    receive file integrity checks only, with domain status not_implemented.

    Return separate tables, source provenance and structured per-table reports.
    Retain source columns/types/order and missingness. Match timing gaps emit a
    warning; statistics gaps are recorded without implying unusable text measures.
    No joins, pivots, feature calculations, normalization or population filtering.

    A supplied existing record pins the version; a new record is written only
    after ALL requested reads and validations succeed. Records stay outside the
    source root, are never overwritten, and use the same format as the single-
    table loader. With no record, select the currently published version once.
    A concurrent first record creation can raise FileExistsError; retry to reuse it.

    Wrong tables argument type raises TypeError; empty/repeated/invalid names or
    missing validation dependencies raise ValueError. Undeclared tables raise
    KeyError before reading table files. Missing files raise FileNotFoundError;
    integrity/structural failures raise ValueError. No partial bundle is returned
    and no new record is saved on read/validation failure. Other I/O errors propagate.
    Memory use includes every loaded DataFrame plus each reader's temporary bytes.
    """
    if isinstance(tables, (str, bytes)) or not isinstance(tables, Sequence):
        raise TypeError("tables must be an ordered sequence of table names, such as ['matches']")
    names = tuple(tables)
    if not names or any(not isinstance(name, str) or not name.strip() for name in names):
        raise ValueError("tables must contain at least one nonempty table name")
    if len(set(names)) != len(names):
        raise ValueError("tables must not contain repeated names")
    if "statistics" in names and "matches" not in names:
        raise ValueError("Selecting statistics requires matches for reference validation")

    root = Path(data_root).resolve()
    path = Path(record_path).resolve() if record_path is not None else None
    if path is not None and path.is_relative_to(root):
        raise ValueError("record_path must be outside data_root to preserve source exports")
    reuse_record = path is not None and path.exists()
    resolved = (_read_season_record(root, season_stem, path) if reuse_record
                else resolve_season_export(root, season_stem))
    absent = [name for name in names if name not in resolved.manifest["tables"]]
    if absent:
        available = ", ".join(sorted(resolved.manifest["tables"]))
        raise KeyError(f"Tables not declared: {', '.join(absent)}; available: {available}")

### 3. Read separate tables and run the available validators

All requested tables are read in request order, preserving their rows, physical columns, Arrow dtypes and missingness. They are not joined or pivoted. Matches receive `validate_matches`; when selected, statistics receive `validate_statistics` against those loaded matches. Statistics validation therefore preserves the full `(event_id, period, group_name, key, side)` grain and checks event/team references automatically here.

Missing or invalid match timing emits a warning and retains all rows, with exact event IDs in `season.validation['matches']`. Statistics gaps are recorded in their report without a warning or automatic filling. Other tables, including pregame and shots, receive **file integrity checks only** and report domain status `not_implemented`. This is stronger than inspecting a schema/footer, but it does not validate their football meaning or cross-table references.

In [4]:
display(Code(cp7_load_source[cp7_reads_start:cp7_provenance_start], language='python'))
display(pd.Series(cp7_season.validation['matches'], name='match_validation'))
display(pd.Series(cp7_season.validation['statistics'], name='statistics_validation'))
print('Pregame:', cp7_season.validation['pregame'])

frames = {name: read_season_table(resolved, name) for name in names}
    validation = {name: {"status": "not_implemented", "table": name} for name in names}
    if "matches" in frames:
        report = validate_matches(frames["matches"], resolved)
        validation["matches"] = {"status": "structure_valid", **asdict(report)}
        missing_count = len(report.missing_kickoff_event_ids)
        invalid_count = len(report.invalid_kickoff_event_ids)
        if missing_count or invalid_count:
            warnings.warn(
                f"Match timing needs attention: {missing_count} missing and "
                f"{invalid_count} invalid kickoff values; rows retained. "
                "See season.validation['matches'] for event IDs.",
                UserWarning, stacklevel=2,
            )
    if "statistics" in frames:
        report = validate_statistics(frames["statistics"], frames["matches"])
        validation["statistics"] = {"status": "structure_valid", **asdict(report)}

status                       structure_valid
rows                                     380
missing_kickoff_event_ids                 ()
invalid_kickoff_event_ids                 ()
Name: match_validation, dtype: object

status                            structure_valid
rows                                        98224
matches_referenced                            380
team_id_column_present                       True
missing_team_id_row_positions                  ()
missing_value_row_positions                    ()
match_sides_without_statistics                 ()
Name: statistics_validation, dtype: object

Pregame: {'status': 'not_implemented', 'table': 'pregame'}


### 4. Copy metadata and write a new selection only after success

The shared provenance includes the season stem, requested names, version, manifest path, fingerprints and captured manifest. Each frame receives an independent deep copy of the source metadata and its validation report in `frame.attrs['xdiyo']`.

The final source section calls `_write_season_record` only after every read and available validation succeeds. Missing/corrupt files, invalid late statistics or malformed matches produce no bundle and no new record or record parent directory. A write failure after verification can still fail the call; this is not an all-files transaction.

The shared helper serializes the same version-1 selection record used by `load_season_table`: a saved publication plus fingerprints, not copies of the tables. Opening with `"x"` prevents overwriting an existing record, including a concurrent first creator. Records from either loader can be replayed by the other. Earlier checkpoint source displays retain the pre-extraction implementation intentionally.

In [5]:
display(Code(cp7_load_source[cp7_provenance_start:], language='python'))
display(Code(inspect.getsource(_write_season_record), language='python'))
display(pd.Series({key: cp7_season.provenance[key]
                   for key in ('season_stem', 'loaded_tables', 'version', 'manifest_path')},
                  name='shared_selection'))
assert all(frame.attrs['xdiyo']['source']['manifest_sha256'] == cp7_season.provenance['manifest_sha256']
           for frame in cp7_season.tables.values())

provenance = {
        "season_stem": season_stem, "loaded_tables": names,
        "version": resolved.manifest["version"],
        "manifest_path": str(resolved.manifest_path),
        "publication_sha256": resolved.publication_sha256,
        "manifest_sha256": resolved.manifest_sha256,
        "manifest": deepcopy(resolved.manifest),
    }
    for name, frame in frames.items():
        # Keep frame attributes independent of the bundle's mutable dictionaries.
        frame.attrs["xdiyo"] = {
            "source": {**deepcopy(provenance), "table": name},
            "validation": deepcopy(validation[name]),
        }
    if path is not None and not reuse_record:
        _write_season_record(resolved, season_stem, path)
    return SeasonData(tables=frames, provenance=provenance, validation=validation)

def _write_season_record(resolved: ResolvedSeason, season_stem: str, path: Path) -> None:
    """Create a selection record after the caller has verified its entire load."""
    record = {
        "schema_version": 1, "season_stem": season_stem,
        "publication_base64": base64.b64encode(resolved.publication_bytes).decode("ascii"),
        "publication_sha256": resolved.publication_sha256,
        "manifest_sha256": resolved.manifest_sha256,
    }
    payload = json.dumps(record, indent=2) + "\n"
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("x", encoding="utf-8") as stream:
        stream.write(payload)

season_stem                                   Premier_League_24_25
loaded_tables                       (matches, statistics, pregame)
version                           6d950fa327de4fc29c796c7cca101606
manifest_path    C:\Users\luisi\Documents\Programming\Python\xD...
Name: shared_selection, dtype: object

### 5. Read the real diagnostics without overstating coverage

This saved selection returns **380 × 35 matches**, **98,224 × 18 statistics** and **740 × 8 pregame rows**. The match and statistics reports show no timing, team-ID, value or side-coverage gaps. Pregame is not domain-validated by this function.

The separate descriptive check below finds pregame rows for 370 of the 380 matches, leaving 20 match sides without a pregame row. Those absences are visible without changing the match population or inventing positions. Nonmissing positions among existing rows do not establish a complete league table or prove that values were available before a historical prediction cutoff.

In [6]:
cp7_match_sides = {(int(event), side) for event in cp7_season.matches['event_id']
                   for side in ('home', 'away')}
cp7_pregame_sides = {(int(event), side) for event, side in
                     cp7_season.pregame[['event_id', 'side']].itertuples(index=False, name=None)}
display(pd.DataFrame([{
    'reference_matches': len(cp7_season.matches),
    'pregame_rows': len(cp7_season.pregame),
    'matches_with_pregame_rows': cp7_season.pregame['event_id'].nunique(),
    'match_sides_without_pregame_rows': len(cp7_match_sides - cp7_pregame_sides),
    'missing_positions_in_present_rows': int(cp7_season.pregame['position'].isna().sum()),
}]))
display(cp7_season.pregame[['event_id', 'side', 'team_id', 'position']].head(4))

,reference_matches,pregame_rows,matches_with_pregame_rows,match_sides_without_pregame_rows,missing_positions_in_present_rows
0,380,740,370,20,0


,event_id,side,team_id,position
0,12436410,home,50,12.0
1,12436410,away,42,3.0
2,12436419,home,7,15.0
3,12436419,away,38,4.0


### 6. Frozen fields, independent metadata and snapshot reports

This example edits a **deep copy** of the returned bundle. Reassigning `.tables` fails, while editing a nested provenance dictionary or a DataFrame is allowed. Editing bundle provenance does not update a frame's independently copied attributes. A data edit also leaves validation reports unchanged: revalidate after changing inputs if you need a report of the new state.

Copied attributes reduce accidental coupling; they are not an immutable experiment archive. Pandas operations may discard attributes, and provenance still describes the source selection after a caller edits the data. The saved record pins source metadata and is likewise not a copy of transformed data.

In [7]:
cp7_demo = deepcopy(cp7_season)
try:
    cp7_demo.tables = {}
except FrozenInstanceError as error:
    print('Field reassignment:', type(error).__name__, str(error))
else:
    raise AssertionError('SeasonData fields should be frozen')

cp7_demo.provenance['manifest']['scope']['league_name'] = 'edited copy'
print('Edited bundle metadata:', cp7_demo.provenance['manifest']['scope']['league_name'])
print('Independent frame metadata:', cp7_demo.matches.attrs['xdiyo']['source']['manifest']['scope']['league_name'])
cp7_demo.statistics.loc[0, 'value'] = None
print('Edited DataFrame missing values:', int(cp7_demo.statistics['value'].isna().sum()))
print('Original report positions remain:', cp7_demo.validation['statistics']['missing_value_row_positions'])
assert cp7_demo.validation == cp7_validation_before
assert cp7_season.provenance == cp7_provenance_before

Field reassignment: FrozenInstanceError cannot assign to field 'tables'
Edited bundle metadata: edited copy
Independent frame metadata: Premier_League
Edited DataFrame missing values: 1
Original report positions remain: ()


### 7. Distinguish unselected tables from invalid requests

These intentional request errors are caught and displayed in full. Selecting statistics alone fails its dependency check; an unknown table fails availability prechecking. None is returned for an unrequested convenience property, not for a failed table load.

The tests also exercise late missing/corrupt files and structural failures: those raise `FileNotFoundError` or `ValueError` and leave no new record/directory. This notebook does not corrupt source files to demonstrate them.

In [8]:
cp7_requests = [
    ('bare string', 'matches'), ('empty selection', []),
    ('duplicate names', ['matches', 'matches']),
    ('missing reference dependency', ['statistics']),
    ('undeclared table', ['matches', 'standings']),
]
cp7_errors = []
for cp7_label, cp7_tables in cp7_requests:
    try:
        load_season(cp7_data_root, 'Premier_League_24_25', tables=cp7_tables, record_path=cp7_record)
    except (TypeError, ValueError, KeyError) as error:
        cp7_errors.append({'case': cp7_label, 'type': type(error).__name__, 'message': str(error)})
    else:
        raise AssertionError(f'Expected request error: {cp7_label}')
with pd.option_context('display.max_colwidth', None):
    display(pd.DataFrame(cp7_errors))

,case,type,message
0,bare string,TypeError,"tables must be an ordered sequence of table names, such as ['matches']"
1,empty selection,ValueError,tables must contain at least one nonempty table name
2,duplicate names,ValueError,tables must not contain repeated names
3,missing reference dependency,ValueError,Selecting statistics requires matches for reference validation
4,undeclared table,KeyError,"'Tables not declared: standings; available: availability, coverage, goal_actions, goal_sequences, heatmap_points, incidents, lineup_players, lineup_teams, matches, passing_edges, player_statistics, pregame, shots, statistics, timing'"


### 8. Keep the single-table and bundle APIs distinct

`load_season_table(..., table='statistics')` continues to return one verified DataFrame with domain status `not_implemented`. It does not load reference matches or automatically run a cross-table validator. Use standalone `validate_statistics(statistics, matches)` when managing separate loads yourself.

`load_season(..., tables=('matches', 'statistics'))` now performs that validation automatically and retains the report in both the bundle and copied frame attributes. The example below reuses the existing record and confirms that the same statistic values are returned through either API; their validation metadata describes the different work each API performed. A loader-created record from either API is compatible in both directions, as covered by the focused tests.

In [9]:
cp7_single_statistics = load_season_table(
    cp7_data_root, 'Premier_League_24_25', table='statistics', record_path=cp7_record,
)
pd.testing.assert_frame_equal(cp7_single_statistics, cp7_season.statistics)
print('Single-table status:', cp7_single_statistics.attrs['xdiyo']['validation']['status'])
print('Bundle status:', cp7_season.validation['statistics']['status'])
assert (cp7_single_statistics.attrs['xdiyo']['source']['manifest_sha256']
        == cp7_season.provenance['manifest_sha256'])
print('Same selected source and data; separate validation coverage.')

Single-table status: not_implemented
Bundle status: structure_valid
Same selected source and data; separate validation coverage.


### 9. Confirm preservation and this batch's fingerprints

The checks compare the original bundle with its saved copies and recheck all selected file/record hashes and all eight analytics source files. The deliberate in-memory edits above were confined to a copy. Version-consistency and record-write behavior are tested with synthetic exports in `tests/analytics/test_season.py`; no collected files are altered.

The full analytics suite includes the existing single-table tests, so the helper extraction is checked alongside the new API. Future source changes require new evidence; a previous batch's recorded hashes are not silently treated as current.

In [10]:
for cp7_name, cp7_frame in cp7_season.tables.items():
    pd.testing.assert_frame_equal(cp7_frame, cp7_frames_before[cp7_name])
    assert cp7_frame.attrs == cp7_frames_before[cp7_name].attrs
assert cp7_season.provenance == cp7_provenance_before
assert cp7_season.validation == cp7_validation_before
cp7_input_after = {name: hashlib.sha256((cp7_root / name).read_bytes()).hexdigest() for name in cp7_input_before}
assert cp7_input_before == cp7_input_after == cp7_evidence['input_sha256_after']
cp7_source_after = {name: hashlib.sha256((cp7_root / name).read_bytes()).hexdigest() for name in cp7_expected_source}
assert cp7_source_before == cp7_source_after == cp7_expected_source
print('Source data, saved record, original bundle and checked library state preserved.')
for cp7_name in ('src/xdiyo_analytics/data/season.py', 'src/xdiyo_analytics/data/loading.py',
                 'src/xdiyo_analytics/data/__init__.py'):
    print(cp7_name, 'SHA256:', cp7_source_after[cp7_name])

Source data, saved record, original bundle and checked library state preserved.


src/xdiyo_analytics/data/season.py SHA256: 39beaa1364b3a52b529ebaccfb4e0064db3a448fc7a66768f2716ec1814c9ca4
src/xdiyo_analytics/data/loading.py SHA256: 6db4b88170229571cc4b76cfa2d1b130de665674069cfe29ee5c9f7517e11da1
src/xdiyo_analytics/data/__init__.py SHA256: 577a8b37cadebfa41c53d953ecc7cb8d65a6da6d14e614651c5694ebbb67700e


## Verification and next discussion

**311 analytics tests passed, including 54 new season-bundle cases.** They cover one resolution with a changing/removed publication pointer, stable references, ordered exact selections, prechecks and dependencies, automatic validation, late failures without a saved record, warning/gap preservation, copied metadata, mutable report snapshots, both record replay directions and exclusive creation. All earlier analytics tests passed after extracting the shared writer.

The saved real load returned **matches 380 × 35; statistics 98,224 × 18; pregame 740 × 8**. Matches/statistics passed structural validation with no reported gaps. Pregame domain validation remains `not_implemented`; the descriptive check found 20 match sides without pregame rows. Shots were not requested.

**Only these 21 new cells (10 code cells) were executed in a fresh `misc314_py314` kernel.** All 102 earlier cells, outputs and notebook metadata were preserved; the notebook now contains 123 cells. This was not a full-notebook or D-Tale rerun. Earlier source/fingerprint outputs remain historical, including the pre-extraction single-table loader and prior Checkpoint 6 fingerprint baseline.

**Next discussion:** choose each model's tables and period/group/key selections and define required completeness/timing policies before joins. This bundle does not join data, fill gaps, create features, select experiment leagues, validate pregame/shots domain semantics or prove historical prediction-time availability. Memory use includes every selected DataFrame and each reader's temporary compressed bytes.